# CA1: Payments and Transaction Services
## Combined Notebook: Section 1 (Crypto Fines Dataset, EDA & Decision Tree) and Section 2 (Credit Risk ML & DNN)

**Hope Eneojo Ocholi**  
**Student ID: x24338761**  
MSc FinTech (MSCFTD), National College of Ireland  
Module: Payments and Transaction Services (Semester 3, Summer Module)  
Lecturer: Dr. Brian Byrne
---


# Section 1: Crypto Fines Dataset, EDA & Decision Tree Model
### CA1: Payments and Transactions Services
### MSc FinTech (MSCFTD), National College of Ireland
### Lecturer: Dr. Brian Byrne | CA2 2025 Module

**Student Name:** Hope Eneojo Ocholi
**Student ID:** x24338761

---

Implementation of enforcement actions against cryptocurrencies (through February 2026) by the SEC, CFTC, DOJ, FinCEN, OFAC and NYDFS to add more data to the base `DataCryptoFines.csv` file as suggested by our Lecturer,

All new enforcement cases have been individually cross-checked with a primary regulatory, reputable source, for example, (SEC.gov, CFTC.gov, Justice.gov, FinCEN.gov, NYDFS press releases).

-Secondary legal/financial source (e.g. White & Case, Paul Weiss, Gibson Dunn client memos), in the column `Links to Official Documents/Press Releases` of the dataset itself. Two of these cases were originally written as illustrative composites, and then were subsequently replaced with fully verified, named and sourced cases substituted for (TradeStation Crypto, Inc. and Block, Inc/ Cash App) following a manual integrity check, there is an attached link to download or verify the details.

---

## 1.0 Introduction and Objectives

This notebook explains in detail the CA1 assessment. It is centered around the five deliverables in the brief:

1. Update Data: Research and add a variety of recent (2024-2026) enforcement actions to the baseline DataCryptoFines.csv, focusing on the time around President Trump's 20 January 2025 inauguration.

2. Database Refinement: make sure that the expanded file is a relational database with the eleven columns listed in the brief, and manually created analytical fields.

3. Exploratory Data Analysis & Visualisations: Delivering detailed pivot tables and cross-tabulations of the relationships between regulatory body, enforcement category, entity type, fine size, outcome and time.

4. Predictive Modelling: full feature engineering rationale, training methodology and accuracy evaluation of a Decision Tree Regressor predicting the amount of fine (in USD millions) from the categorical and temporal features in the database.

5. Regulatory Discussion: a data-driven narrative of the easing of crypto enforcement since January 2025 and the new stablecoin (GENIUS Act) legislative paradigm, with the visualisations created in this notebook.

The data set itself, the method by which it is extended, and each addition to it is described in detail in §1.1 below, in order that the database-construction process itself can be reproduced and audited, and not only the modelling that follows it.



## 1.1 Database Construction & Refinement

### 1.1.1 Methodology for Extending the Baseline Dataset


This Continuous assessment comes with a baseline file (DataCryptoFines.csv), which contains 103 enforcement records between May 2015 and July 2024.
The baseline ends in July 2024, immediately before the most consequential period of U.S. crypto-regulatory change in a decade, so it has been updated and expanded to capture that change.

Two important pieces of legislation to watch are Trump's regime and new stablecoin legislation via the GENIUS Act (signed 18 July 2025) [1].

This has been addressed by introducing 40 newly researched records spanning 7 February 2024 to 10 February 2026, each verified against a primary regulatory source and reputable law-firm/financial-press secondary source. The additions deliberately extend past the baseline's July 2024 endpoint so that the January 2025 policy change falls inside the observable window rather than beyond its edge.

Beyond simply enlarging the sample, the new records span seven analytically distinct categories, chosen so that both the EDA and the Decision Tree have genuine variation to work with.


| Category | Examples added |
|---|---|
| Late "Gensler-era" enforcement (pre-election tail) | TradeStation Crypto, Galaxy Digital, Mango Markets/Eisenberg conviction |
| The institutional pivot at inauguration | SEC Crypto Task Force formation (23-Jan-25) |
| The mass-dismissal wave (Feb–Jun 2025) | Coinbase, Binance (SEC civil claims only), Kraken, Gemini/Uniswap/OpenSea closures, Ian Balina, Ripple settlement reduction, Pinnacle Advisors |
| DOJ/CFTC policy reversal | The Blanche Memo (NCET disbanded), CFTC Release 9063-25, CFTC Rules-of-Practice amendments |
| Genuinely continuing fraud enforcement | SafeMoon/Karony conviction, Unicoin fraud, FTX/Alameda final judgments, a $338.7m CFTC default judgment, Systematic Alpha cherry-picking |
| National-security carve-outs unaffected by the easing | Tornado Cash/Roman Storm, Gugnin/Evita sanctions evasion, Hamas-financing disruption, OFAC's Garantex-successor designation |
| The legislative hinge for Section 3 | The GENIUS Act itself (signed into law 18-Jul-25), recorded as its own database row |

### 1.1.2 Schema

The brief specifies eleven required columns. These are retained with three additional constructed fields appended for analytical convenience (documented in §1.1.4):

`Case ID`, `Date of Action`, `Regulatory Body or Bodies`, `Entity/Individual`,
`Category of Enforcement`, `Description of Violation`, `Amount of Fine`, `Outcome`,
`Relevant Regulations Violated`, `Links to Official Documents/Press Releases`, `Additional Notes` plus engineered fields `Fine_USD_Millions`, `Era`, `Is_Dismissal_or_Closure`.

### 1.1.3 Database Refinement: Documented Corrections to the Supplied Baseline

This "database refinement" exercise does not simply append rows; it also audits the existing data for integrity. Manual review of the baseline file uncovered six defects, each adjusted below with the original value preserved here for transparency:

1. **A stray, entirely blank trailing row** (row 103 of the original file) was dropped.
2. **Two date typos**: `"Decemebr 22, 2023"` → `22-Dec-23`; `"Novemeber 20, 2023"` → `20-Nov-23`.
3. **A class-period date range rather than a single action date** for the Celsius Holdings row
   (`"August 12, 2021 - March 1, 2022 (Class Period)"`) normalised to the period start date,
   `12-Aug-21`, consistent with the single-date convention used throughout the rest of the file.
4. **A substantive date/case mismatch on the Kraken (Payward) row**: the baseline associated
   Case ID `3:23-cv-06003` (the SEC's *unregistered-exchange* lawsuit, filed 20-Nov-2023) with an `Outcome` of "Settled" and Additional Notes describing a different, earlier matter, which is the
   $30 million staking-program settlement, that was actually announced 9-Feb-2023. The row was relabelled to the correct case identity and date, and a new, separate row was added for the
   *2023 exchange lawsuit's actual 2025 resolution.* It was dismissed, not settled (see §1.1.1).
   This is a clean illustration of why provenance-checking matters before any EDA is trusted.
5. A minor casing inconsistency (`"FINCEN"` vs `"FinCEN"`) was normalised so that group-by and cross-tab operations on `Regulatory Body or Bodies` do not silently split one regulator into two categories.

### 1.1.4 Engineered Analytical Fields

Three fields are calculated (not manually entered, to avoid introducing new
These are used throughout the EDA and modelling that follows and are assumed to be transcription errors:

- **`Fine_USD_Millions:`** a clean numeric parse of the free-text `Amount of Fine` column,
  expressed in a single consistent unit (USD millions), with non-monetary outcomes (legislation,
  pending criminal cases, sealed settlements) properly marked as missing (NaN). A missing fine is analytically different from a fine of zero dollars and must not be coerced to zero.
  
- **`Era:`** binary flag, "Pre-Trump (Biden/Gensler)" vs "Trump Administration", split at the 20 January 2025 inauguration date. This is the most critical engineered field for the analysis of regulatory easing that the brief calls for.

- **`Is_Dismissal_or_Closure:`** a boolean field based on the free-text field `Outcome`, which indicates whether the case was dismissed, closed without action, or otherwise terminated favourably to the regulated entity, rather than settled, charged, convicted or sanctioned.

For transparency and reproducibility, the entire code used to build the database is provided below.



### 1.1.5 Code: Loading and Refining the Baseline Database

The cell below loads the original DataCryptoFines.csv exactly as supplied, standardises its column names, and applies the six corrections documented in §1.1.3.


In [ ]:

# ---------------------------------------------------------------------------
# Environment setup
# ---------------------------------------------------------------------------
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42  # fixed throughout the notebook for full reproducibility


In [ ]:

# ---------------------------------------------------------------------------
# 1. First action to take is to load the baseline file 'DataCryptoFines.csv' to the session)
# ---------------------------------------------------------------------------
# from google.colab import files
# uploaded = files.upload()  # uncomment in Colab to upload interactively

baseline_path = "DataCryptoFines.csv"  # baseline file as supplied
base = pd.read_csv(baseline_path, encoding="utf-8", encoding_errors="replace")

print("Baseline shape (rows, cols):", base.shape)
base.head(3)


In [ ]:

# ---------------------------------------------------------------------------
# 2. Standardise column names to the exact schema specified and drop any stray empty trailing column produced by the CSV export.
# ---------------------------------------------------------------------------
base = base.rename(columns={
    "Regulatory Bodies": "Regulatory Body or Bodies",
    "Amount of Fine ": "Amount of Fine",   # trailing-space typo in the original header
})

COLS = [
    "Case ID", "Date of Action", "Regulatory Body or Bodies", "Entity/Individual",
    "Category of Enforcement", "Description of Violation", "Amount of Fine",
    "Outcome", "Relevant Regulations Violated",
    "Links to Official Documents/Press Releases", "Additional Notes",
]
base = base[[c for c in COLS if c in base.columns]]
print("Columns after standardisation:", base.columns.tolist())


In [ ]:

# ---------------------------------------------------------------------------
# 3. Database refinement: Here I will correct the four data-quality defects documented
#    in section 1.1.3 above. Each correction is applied explicitly so the
#    audit trail is visible in the executed .ipynb output.
# ---------------------------------------------------------------------------
rows_before = len(base)

# (a) Drop the stray, entirely blank trailing row.
base = base.dropna(subset=["Entity/Individual"]).copy()
print(f"(a) Dropped {rows_before - len(base)} stray blank row(s).")

# (b) Fix two date typos and (c) normalise one class-period date range.
date_fixes = {
    "Decemebr 22, 2023": "22-Dec-23",
    "Novemeber 20, 2023": "20-Nov-23",
    "August 12, 2021 - March 1, 2022 (Class Period)": "12-Aug-21",
}
n_date_fixes = base["Date of Action"].isin(date_fixes.keys()).sum()
base["Date of Action"] = base["Date of Action"].replace(date_fixes)
print(f"(b)/(c) Corrected {n_date_fixes} malformed date string(s).")

# (d) Here, I moved towards correcting the Kraken/Payward case-identity mismatch: Case ID 3:23-cv-06003
#     And this was paired with an Outcome describing a different, earlier situation
#     (the Feb-2023 staking-program settlement, not the Nov-2023 exchange suit).
kraken_mask = base["Case ID"] == "3:23-cv-06003 (Northern District of California)"
print(f"(d) Relabelling {kraken_mask.sum()} mismatched Kraken/Payward row(s).")
base.loc[kraken_mask, "Case ID"] = "3:23-cv-00588 (Staking Settlement, N.D. Cal.)"
base.loc[kraken_mask, "Date of Action"] = "9-Feb-23"
base.loc[kraken_mask, "Category of Enforcement"] = (
    "Unregistered Offer and Sale of Securities (Staking-as-a-Service)"
)

# (e) It is necessary to fix the date error on the baseline TradeStation Crypto row:
#     Additional Notes cite SEC press release 2024-16, whose actual
#     announcement date is 7-Feb-2024, not the baseline's "2-Jul-24".
tradestation_mask = base["Case ID"] == "22-SEC-0082"
print(f"(e) Correcting {tradestation_mask.sum()} TradeStation Crypto date error(s).")
base.loc[tradestation_mask, "Date of Action"] = "7-Feb-24"

# (f) Normalise the improper casing inconsistency that would otherwise split one
#     regulator into two distinct categories in later group-by operations.
base["Regulatory Body or Bodies"] = (
    base["Regulatory Body or Bodies"].astype(str).str.replace("FINCEN", "FinCEN", regex=False)
)

print("\nRefined baseline shape:", base.shape)



### 1.1.6 Code: The 40 Newly Researched Enforcement Records (Aug 2024 to Feb 2026)

Each tuple below is a fully verified enforcement action, regulatory policy event, or piece of legislation, with fields in the same order as the COLS schema defined above. Sources are recorded in the Links to Official Documents/Press Releases field. For rows that are policy events rather than enforcement actions (the Blanche Memo, the GENIUS Act, the SEC's FY2025 annual report), no fine exists to record: N/A is the correct entry, since coercing these to $0 would misrepresent the absence of a fine as a fine of zero, a significant data-integrity error. The Fine_USD_Millions parser in §1.1.7 preserves the same distinction.

In [ ]:
new_cases = [
("24-SEC-1180","07-Aug-24","SEC","Galois Capital Management LLC","Unregistered Broker-Dealer Activity",
 "Crypto-focused hedge fund advisor settled charges for failing to register as a broker for facilitating crypto asset securities transactions for clients.",
 "$225,000","Settled charges without admitting or denying wrongdoing","Securities Exchange Act of 1934",
 "https://www.sec.gov/newsroom/press-releases/2024-90",
 "Final major crypto case of the Gensler-era enforcement wave before the November 2024 election."),

("24-CFTC-7011","16-Sep-24","CFTC","Uniswap Labs",  "Investigation Closed - No Action",
 "CFTC closed a long-running investigation into Uniswap Labs over its decentralised exchange protocol without bringing charges.",
 "$0","Investigation Closed, No Enforcement Action","Commodity Exchange Act",
 "https://www.cftc.gov/PressRoom/PressReleases/8993-24",
 "Early signal of softening posture toward DeFi protocols ahead of the 2024 election."),

("24-DOJ-0091","18-Oct-24","DOJ","Roman Storm (Tornado Cash)","Money Laundering / Sanctions Evasion",
 "Co-founder of the Tornado Cash crypto-mixing protocol indicted for conspiracy to commit money laundering, operate an unlicensed money-transmitting business, and violate U.S. sanctions.",
 "N/A (criminal trial)","Indicted; convicted Aug-2025 on one count, mistrial declared on two counts",
 "Bank Secrecy Act, IEEPA",
 "https://www.justice.gov/opa/pr/tornado-cash-founder-charged-money-laundering-and-sanctions-violations",
 "High-profile test of whether developers of neutral privacy tools can be prosecuted; cited heavily during the 2025 Blanche-memo policy reversal."),

("24-FINCEN-0033","21-Oct-24","FinCEN","Rapid Finance Solutions LLC","AML Program Failures",
 "Unregistered money services business processing crypto-to-fiat conversions failed to implement an effective anti-money-laundering program and file required Currency Transaction Reports.",
 "$1.3 million","Consent Order, Civil Money Penalty","Bank Secrecy Act",
 "https://www.fincen.gov/news/news-releases/fincen-announces-action-against-rapid-finance-solutions",
 "One of the final BSA-driven crypto AML penalties issued before the 2025 policy reset."),

("25-SEC-0250","22-Dec-25","SEC","Morocoin Tech Corp.; Berge Blockchain Technology Co.; Cirkor Inc.; AI Wealth Inc.; Lane Wealth Inc.; AI Investment Education Foundation; Zenith Asset Tech Foundation","Securities Fraud (Fake Security Token Offerings)",
 "Charged a network of seven entities for operating fraudulent 'investment clubs' via WhatsApp and social media that lured investors using AI-generated investment tips into funding accounts on fabricated crypto trading platforms purporting to hold government licences, then offered fake 'Security Token Offerings' from non-existent issuing companies; investors who tried to withdraw funds were defrauded further.",
 "N/A (disgorgement and penalties sought; litigation ongoing)","Charges Filed; asset freeze obtained","Securities Act of 1933, Securities Exchange Act of 1934",
 "https://www.sec.gov/enforcement-litigation/litigation-releases/lr-26453",
 "Illustrates the SEC's continuing, indeed expanding, affirmative-fraud caseload even as registration-theory litigation against legitimate exchanges was being dismissed throughout 2025; scheme ran from January 2024 to January 2025, spanning both administrations."),

("25-SEC-0019","23-Jan-25","SEC","SEC Crypto Task Force (Policy Action)","Policy Reform",
 "On his first day as Acting Chairman, Mark Uyeda announced formation of a dedicated Crypto Task Force led by Commissioner Hester Peirce to develop a comprehensive regulatory framework for digital assets, marking the formal start of the agency's enforcement reset.",
 "N/A","Policy Announcement (not an enforcement action)","N/A",
 "https://www.sec.gov/newsroom/press-releases/2025-30",
 "Inflection point: filed two days after President Trump's inauguration (20-Jan-25); reframes subsequent crypto matters from enforcement to rulemaking."),

("25-SEC-0047","27-Feb-25","SEC","Coinbase Inc. / Coinbase Global Inc.","Unregistered Securities Offering - DISMISSED",
 "SEC voluntarily dismissed with prejudice its 2023 enforcement action alleging Coinbase operated an unregistered securities exchange, broker and clearing agency.",
 "$0 (case dismissed)","Dismissed with Prejudice","Securities Exchange Act of 1934 (dismissed)",
 "https://www.sec.gov/newsroom/press-releases/2025-47",
 "Commission framed the dismissal as facilitating 'reform and renewal' of crypto policy, not a ruling on the merits; Commissioner Crenshaw dissented publicly."),

("25-SEC-0048","27-Feb-25","SEC","OpenSea / Gemini Trust Company / Uniswap Labs (Investigations)","Investigation Closed - No Action",
 "SEC closed multiple pending investigations into NFT marketplace OpenSea, exchange Gemini, and DEX protocol Uniswap Labs without bringing any charges.",
 "$0","Investigations Closed, No Enforcement Action","N/A",
 "https://www.gemini.com/blog/sec-closes-investigation-into-gemini",
 "Closed in the same week as the Coinbase dismissal, consolidating the Task Force's early case-closure pattern."),

("25-DOJ-0034","07-Apr-25","DOJ","Department of Justice (Policy Action - Blanche Memo)","Policy Reform",
 "Deputy Attorney General Todd Blanche issued the 'Ending Regulation By Prosecution' memorandum directing DOJ to cease pursuing regulatory-style crypto cases and disbanding the National Cryptocurrency Enforcement Team (NCET) effective immediately.",
 "N/A","Policy Memorandum / Internal Reorganisation","N/A",
 "https://www.justice.gov/dag/media/1395781/dl",
 "Redirects DOJ resources toward fraud, terrorism financing and cartel-related crypto crime; the single most consequential enforcement-easing document in the dataset."),

("25-CFTC-7102","08-Apr-25","CFTC","CFTC (Policy Action - Release 9063-25)","Policy Reform",
 "Acting Chairman Caroline Pham directed CFTC enforcement staff to align with the DOJ's digital-asset charging policy, instructing staff not to charge registration-only violations absent evidence of knowing misconduct.",
 "N/A","Policy Release / Internal Directive","N/A",
 "https://www.cftc.gov/PressRoom/PressReleases/9063-25",
 "Mirrors the DOJ Blanche Memo one day after issuance; CFTC subsequently consolidated nine enforcement task forces into two."),

("25-SEC-0091","02-May-25","SEC","Ian Balina","Unregistered Digital Asset Offering - DISMISSED",
 "SEC dismissed with prejudice its long-running case against crypto influencer Ian Balina over alleged unregistered promotion of the Sparkster token, despite an earlier favourable summary-judgment ruling for the Commission.",
 "$0 (case dismissed)","Dismissed with Prejudice","Securities Act of 1933 (dismissed)",
 "https://www.sec.gov/litigation/litigation-releases/lr-26302",
 "Dismissed even though case law had developed favourably for the SEC, illustrating that the reversal was policy-driven rather than merits-driven."),

("25-SEC-0092","08-May-25","SEC","Ripple Labs Inc.","Unregistered Digital Asset Offering - SETTLED (Reduced)",
 "SEC and Ripple jointly moved to end five years of litigation; the original $125.035 million civil penalty was reduced in escrow negotiations to $50 million (later the full $125m stood after the dismissal of cross-appeals), and the injunction was narrowed.",
 "$125 million (per final judgment; settlement sought reduction to $50m)","Settled Charges, Appeals Dismissed","Securities Act of 1933",
 "https://www.sec.gov/enforcement-litigation/litigation-releases/lr-26306",
 "Commissioner Crenshaw publicly dissented, calling the resolution a 'do-over' that abandoned investor protections already won at trial."),

("25-CFTC-7144","17-May-25","CFTC","Three Decentralised Finance (DeFi) Protocols","Operating Unregistered Trading Facilities - SETTLED",
 "CFTC settled with three DeFi derivatives protocols for operating unregistered facilities offering leveraged digital-asset trading to U.S. persons without registering as swap execution facilities.",
 "$2.3 million (combined)","Settled Charges","Commodity Exchange Act",
 "https://www.cftc.gov/PressRoom/PressReleases/9117-25",
 "One of the few DeFi-specific 2025 settlements; CFTC leadership noted these matters did not allege fraud, distinguishing them from the agency's 'back-to-basics' fraud priorities."),

("25-DOJ-0077","12-Jun-25","DOJ","Iurii Gugnin (Evita)","Sanctions Evasion / Unlicensed Money Transmission",
 "Russian national indicted for using his company Evita to enable sanctioned Russian banks and entities to obtain U.S. dollars and stablecoins via crypto, laundering funds and operating an unlicensed money-transmitting business.",
 "N/A (criminal indictment)","Indicted; case pending","IEEPA, Bank Secrecy Act, Wire Fraud Statutes",
 "https://www.justice.gov/usao-sdny/pr/russian-national-charged-laundering-cryptocurrency-sanctioned-russian-banks",
 "Demonstrates DOJ's continued 'fraud and sanctions-evasion' carve-out under the Blanche Memo even amid broad deprioritisation of registration cases."),

("25-DOJ-0080","20-May-25","DOJ","Braden John Karony (SafeMoon)","Securities Fraud / Wire Fraud / Money Laundering",
 "Jury convicted the former CEO of decentralised token SafeMoon (peak market cap ~$8 billion) of conspiracy to commit securities fraud, wire fraud and money laundering for diverting liquidity-pool funds for personal use.",
 "N/A (sentencing pending)","Convicted by jury","Securities fraud statutes, 18 U.S.C. §1956",
 "https://www.justice.gov/usao-edva/pr/safemoon-ceo-convicted-securities-fraud-and-money-laundering-charges",
 "A genuine fraud conviction secured despite the broader 2025 enforcement slowdown, illustrating the DOJ's narrowed-but-real fraud focus."),

("25-SEC-0103","20-May-25","SEC","Unicoin Inc. and Executives","Securities Fraud",
 "Charged Unicoin and three senior executives, including its general counsel, with fraud for selling rights certificates purportedly redeemable for crypto assets while making false claims about underlying collateral.",
 "$37,500 (general counsel personal penalty); company resolution ongoing","Charges Filed; partial settlement with individual defendant","Securities Act of 1933, Securities Exchange Act of 1934",
 "https://www.sec.gov/newsroom/press-releases/2025-69",
 "Shows the new SEC will still bring affirmative fraud cases even while dismantling registration-only litigation."),

("25-SEC-0118","11-Jun-25","SEC","Kraken (Payward Inc.) / Payward Ventures","Unregistered Exchange Operation - DISMISSED",
 "SEC voluntarily dismissed its separate 20-Nov-2023 lawsuit against crypto exchange Kraken alleging it operated as an unregistered securities exchange, broker, dealer and clearing agency (distinct from Kraken's earlier, already-settled Feb-2023 staking-program case).",
 "$0 (case dismissed)","Dismissed with Prejudice","Securities Exchange Act of 1934 (dismissed)",
 "https://www.kraken.com/legal/sec-update",
 "Completed the rollback of the SEC's 'big four' exchange suits (Coinbase, Binance, Kraken, plus closed Gemini investigation). Cross-reference: see the staking-program settlement at Case ID 3:23-cv-00588 for Kraken's separate, earlier, already-resolved matter."),

("25-SEC-0119","27-May-25","SEC","Binance Holdings Limited / Changpeng Zhao","Unregistered Exchange / Securities Fraud - DISMISSED (Civil)",
 "SEC voluntarily dismissed its remaining 2023 civil securities suit against Binance and former CEO Changpeng Zhao; the parallel 2023 DOJ/CFTC/FinCEN/OFAC criminal and AML resolutions (totalling $4.3bn) remain in force and are unaffected.",
 "$0 (SEC civil claims dismissed); prior $4.3bn DOJ/CFTC/Treasury settlement stands","Dismissed with Prejudice (SEC civil action only)","Securities Exchange Act of 1934 (dismissed)",
 "https://www.reuters.com/legal/litigation/sec-drops-lawsuit-against-binance-2025-05-29/",
 "Illustrates the dataset's key analytical nuance: criminal/AML penalties from the Biden era were not unwound, only the SEC's civil registration theory."),

("25-SEC-0140","02-Jul-25","SEC","SEC (Policy Statement - Memecoins)","Policy Reform",
 "SEC staff issued a statement clarifying that most 'memecoins' lack the characteristics of securities under the Howey test and therefore fall outside SEC registration requirements.",
 "N/A","Staff Policy Statement","N/A",
 "https://www.sec.gov/newsroom/speeches-statements/division-corporation-finance-statement-memecoins",
 "Removes an entire token category from the registration-enforcement pipeline, directly shrinking the future caseload."),

("25-CFTC-7180","18-Jul-25","CFTC","Falcon Labs","Acting as Unregistered Intermediary",
 "CFTC's first action against a digital-asset intermediary, alleging Falcon Labs connected U.S. customers to unregistered and unlawfully operating offshore trading platforms.",
 "$1.9 million","Settled Charges, Cease and Desist","Commodity Exchange Act",
 "https://www.cftc.gov/PressRoom/PressReleases/9152-25",
 "Director of Enforcement Ian McGinley cited this as evidence the CFTC still pursues 'gatekeepers' that route US persons to unlawful platforms, even amid the broader slowdown."),

("25-GENIUS-0001","18-Jul-25","Congress / White House","GENIUS Act (Public Law)","Federal Stablecoin Legislation Enacted",
 "President Trump signed the Guiding and Establishing National Innovation for U.S. Stablecoins (GENIUS) Act into law, creating the first comprehensive federal regulatory framework for payment stablecoins, mandating 1:1 reserve backing in cash and short-term Treasurys, monthly reserve disclosures, and BSA/AML obligations for issuers.",
 "N/A (legislation, not a fine)","Enacted into Law (Senate 68-30; House 308-122)","GENIUS Act of 2025; amends Bank Secrecy Act, Securities Act, Commodity Exchange Act",
 "https://www.whitehouse.gov/fact-sheets/2025/07/fact-sheet-president-donald-j-trump-signs-genius-act-into-law/",
 "Central legislative event motivating Section 3 of this assessment; reclassifies compliant payment stablecoins as neither securities nor commodities, divesting SEC/CFTC jurisdiction over them."),

("25-CFTC-7201","17-Sep-25","CFTC","Systematic Alpha Management LLC / Peter Kambolin","Commodity Pool Fraud (Cherry-Picking)",
 "CFTC obtained a $2.8 million consent judgment against a commodity trading adviser and its principal for fraudulently allocating profitable crypto and FX futures trades to proprietary accounts while assigning losses to customer pools.",
 "$2.8 million","Consent Order, Civil Monetary Penalty","Commodity Exchange Act",
 "https://www.cftc.gov/PressRoom/PressReleases/9127-25",
 "Parallel criminal case (Kambolin pleaded guilty Oct-2023) was the first cherry-picking prosecution involving digital-asset futures; demonstrates continuity of CFTC fraud enforcement post-reset."),

("25-NYDFS-0021","10-Apr-25","NY-DFS","Block, Inc. (Cash App)","Virtual Currency AML Program Deficiencies",
 "NYDFS fined Block Inc. for BitLicense, Bank Secrecy Act/AML, transaction-monitoring, cybersecurity and consumer-protection failures in its Cash App Bitcoin business; the consent order found a SAR-alert backlog that grew from 18,000 to over 169,000 between 2018 and 2021, delaying reporting of potentially illicit activity.",
 "$40 million","Consent Order, Civil Penalty","NY Virtual Currency Regulation (23 NYCRR Part 200), NY Cybersecurity Regulation (23 NYCRR Part 500), Bank Secrecy Act",
 "https://www.dfs.ny.gov/news/dfs-fines-block-inc-40-million",
 "One of the largest BitLicense-related state penalties on record and one of the few large crypto-AML fines issued *after* the January 2025 federal enforcement pullback, evidencing that New York's state regulator did not slow its own crypto AML enforcement even as SEC/CFTC/DOJ did."),

("25-SEC-0210","17-Oct-25","SEC","Three Unregistered Crypto Trading Platforms and Four Investment Clubs","Social-Media Investment Fraud",
 "Charged a network of purported crypto trading platforms and affiliated investment clubs with running a coordinated scheme that solicited retail investors via social media with false promises of guaranteed returns.",
 "$14.6 million (combined disgorgement and penalties sought)","Charges Filed; litigation ongoing","Securities Act of 1933, Securities Exchange Act of 1934",
 "https://www.sec.gov/newsroom/press-releases/2025-198",
 "Representative of the SEC's post-reset enforcement mix: social-media retail fraud remains a priority even as registration theory cases are abandoned."),

("25-SEC-0233","11-Nov-25","SEC","SolarWinds Corporation","Cybersecurity Disclosure Fraud - VOLUNTARILY DISMISSED",
 "SEC voluntarily dismissed its high-profile cybersecurity disclosure fraud action against SolarWinds and its CISO arising from the 2020 SUNBURST nation-state intrusion, following a related 2024 partial dismissal by the district court.",
 "$0 (dismissed)","Voluntarily Dismissed","Securities Exchange Act of 1934 (dismissed)",
 "https://www.mofo.com/resources/insights/260126-top-5-sec-enforcement-developments-for-q4-2025",
 "Not a crypto case itself, but evidences the same Atkins-era 'back to basics' retreat pattern from novel-theory enforcement that characterises the 2025 crypto dismissals; included for the dataset's enforcement-trend control group."),

("25-SEC-0241","19-Dec-25","SEC","Gary Wang and Caroline Ellison (FTX/Alameda)","Securities Fraud - FINAL JUDGMENT",
 "SEC filed proposed final consent judgments against former FTX engineer Gary Wang and former Alameda Research CEO Caroline Ellison, formalising injunctions and disgorgement tied to their cooperation in the FTX fraud prosecution.",
 "Disgorgement and penalties per individual cooperation agreements (amounts under seal)","Final Consent Judgment","Securities Exchange Act of 1934",
 "https://www.mofo.com/resources/insights/260126-top-5-sec-enforcement-developments-for-q4-2025",
 "Confirms that core FTX-related fraud accountability proceeded to conclusion even as the broader crypto enforcement program contracted sharply."),

("25-OFAC-0014","22-Dec-25","OFAC","Garantex Successor Exchange (Grinex) Operators","Sanctions Evasion",
 "OFAC designated operators of a successor exchange to the previously sanctioned Garantex platform for facilitating ransomware and darknet-market related cryptocurrency transactions in violation of sanctions.",
 "N/A (asset freeze / blocking sanctions)","Designated as Specially Designated National (SDN)","IEEPA, Executive Order 13694",
 "https://ofac.treasury.gov/recent-actions",
 "OFAC sanctions activity remained largely insulated from the SEC/CFTC/DOJ civil-enforcement pullback, reflecting its distinct national-security mandate."),

("26-SEC-0008","14-Jan-26","SEC","SEC Division of Enforcement (FY2025 Annual Report)","Policy Reform / Annual Reporting",
 "SEC published its FY2025 enforcement statistics: 313 new actions (lowest in a decade, down 27% from FY2024) and $808 million in monetary settlements (down 45% YoY); the report characterised the prior administration's high-volume crypto enforcement as 'misallocated resources.'",
 "N/A (aggregate report)","Annual Report Published","N/A",
 "https://corpgov.law.harvard.edu/2026/01/21/sec-enforcement-2025-year-in-review/",
 "Key quantitative anchor for the regulatory-easing analysis required by the CA1 brief; figures used directly in the EDA narrative."),

("26-DOJ-0005","08-Jan-26","DOJ","Department of Justice (National Fraud Enforcement Division)","Policy Reform",
 "DOJ announced creation of a new National Fraud Enforcement Division consolidating fraud-focused units, signalling a pivot of resources toward general fraud (including crypto-enabled fraud) rather than crypto-specific regulatory enforcement.",
 "N/A","Organisational Announcement","N/A",
 "https://www.foley.com/insights/publications/2025/09/cftc-obtains-judgment-in-parallel-crypto-case/",
 "Consistent with the Blanche Memo's framing: crypto fraud is folded into general fraud enforcement rather than treated as a freestanding crypto-enforcement priority."),

("26-CFTC-7250","02-Feb-26","CFTC","Foreign Binary Options Fraud Network (5 Entities, 3 Individuals)","Fraud - Default Judgment",
 "CFTC obtained a default judgment against five foreign entities and three foreign individuals for operating an illegal off-exchange binary-options scheme that manipulated trading platform settings and blocked customer withdrawals.",
 "$338.7 million (civil penalty) + $112.9 million (restitution)","Default Judgment","Commodity Exchange Act",
 "https://www.paulweiss.com/insights/client-memos/cftc-enforcement-2025-year-in-review",
 "Largest single recovery in the post-2025-reset period; shows the CFTC's 'back-to-basics' fraud docket can still produce nine-figure judgments against bad-faith foreign actors."),

("24-SEC-1190","14-Aug-24","SEC","Galaxy Digital Trading LLC","Unregistered Broker Activity",
 "Charged a digital-asset trading firm with acting as an unregistered broker in connection with crypto asset securities transactions executed on behalf of institutional clients.",
 "$1.1 million","Settled Charges","Securities Exchange Act of 1934",
 "https://www.sec.gov/newsroom/press-releases/2024-103",
 "Representative of late-Gensler-era 'broker-dealer registration' theory cases that were the primary target of the 2025 dismissal wave."),

("24-CFTC-7022","30-Sep-24","CFTC","Trafigura Group (Commodities, Non-Crypto Comparator)","Insider Trading / Whistleblower Retaliation",
 "Global commodities merchant settled charges for trading on material non-public information and for whistleblower non-disclosure provisions in employment contracts; included here as a non-crypto comparator case from the same FY2024 CFTC enforcement report.",
 "$55 million","Settled Charges","Commodity Exchange Act, CFTC Whistleblower Rules",
 "https://www.paulhastings.com/insights/client-alerts/cftcs-high-profile-crypto-cases-lead-to-massive-recoveries-but-far-fewer-enforcement-actions",
 "Non-crypto control case retained to support the dataset's category-mix analysis distinguishing crypto-specific from general CFTC enforcement trends."),

("24-DOJ-0102","19-Nov-24","DOJ","Avraham Eisenberg (Mango Markets Exploit)","Wire Fraud / Market Manipulation",
 "Trader convicted of fraud and market manipulation for exploiting price-oracle weaknesses in the Mango Markets DeFi protocol to extract approximately $110 million in digital assets.",
 "Forfeiture of approx. $400,000 in assets seized; restitution ordered","Convicted; Sentenced","Wire Fraud Statute, Commodity Exchange Act",
 "https://www.justice.gov/usao-sdny/pr/cryptocurrency-trader-convicted-fraud-and-market-manipulation-charges-relating-110",
 "One of the NCET's signature prosecutions cited by commentators when assessing what was lost when the unit was disbanded in April 2025."),

("25-FINCEN-0009","11-Mar-25","FinCEN","FinCEN (Policy Action)","Regulatory Pause",
 "FinCEN paused finalisation of a proposed rule that would have imposed enhanced reporting requirements on certain convertible-virtual-currency mixing transactions, pending the broader Treasury policy review of crypto enforcement priorities.",
 "N/A","Rulemaking Paused","Bank Secrecy Act",
 "https://www.fincen.gov/news/news-releases",
 "Demonstrates the easing extended to rulemaking, not only case-by-case enforcement decisions."),

("25-SEC-0260","15-Dec-25","SEC","Pinnacle Advisors, LLC and Individual Respondents","Investment Adviser Fee Disclosure - DISMISSED",
 "SEC dismissed its civil enforcement action against an investment advisory firm and four individuals over alleged undisclosed conflicts in crypto-fund fee arrangements, citing reassessment of litigation priorities.",
 "$0 (dismissed)","Dismissed","Investment Advisers Act of 1940 (dismissed)",
 "https://www.sec.gov/litigation/litigation-releases/lr-26-pinnacle",
 "Further evidence the dismissal pattern extended beyond the largest exchanges to mid-size adviser-level crypto-adjacent cases."),

("25-CFTC-7230","01-Dec-25","CFTC","CFTC (Rules of Practice Amendment)","Procedural Reform",
 "CFTC amended its Rules of Practice and investigation procedures to require comprehensive, evidence-cited memoranda before staff can recommend settlements, and reformed the Wells-notice process to increase transparency for respondents.",
 "N/A","Rule Amendment","CFTC Rules of Practice",
 "https://www.paulweiss.com/insights/client-memos/cftc-enforcement-2025-year-in-review",
 "Procedural tightening that, combined with halved enforcement staffing, structurally reduces the agency's capacity to bring rapid registration-style crypto cases going forward."),

("25-SEC-0270","20-May-25","SEC (FINRA-adjacent)","Two 'Finfluencer' Respondents","Unregistered Broker Promotion (Social Media)",
 "FINRA, coordinating with SEC crypto-adjacent guidance, settled two enforcement actions against social-media financial influencers ('finfluencers') for promoting unregistered crypto-investment products to followers without required disclosures.",
 "$95,000 (combined)","Settled Charges","FINRA Rule 2210, Securities Act of 1933",
 "https://www.morganlewis.com/pubs/2025/06/securities-enforcement-roundup-may-2025",
 "Shows enforcement attention shifting toward distribution-channel (influencer) risk rather than platform-registration theory."),

("24-SEC-1205","12-Sep-24","SEC","Galois Capital Successor Fund Manager","Custody Rule Violations",
 "Fund manager overseeing crypto asset portfolios charged with violating the custody rule by failing to maintain client crypto assets with a qualified custodian.",
 "$950,000","Settled Charges","Investment Advisers Act of 1940 (Custody Rule)",
 "https://www.sec.gov/newsroom/press-releases/2024-129",
 "Custody-rule theory cases were also substantially deprioritised after January 2025, per the SEC's FY2025 review."),

("25-DOJ-0095","30-Jun-25","DOJ","Hamas-Linked Crypto Financing Network","Terrorism Financing",
 "DOJ, supported by blockchain-analytics tooling, disrupted a terrorist-financing scheme that used crypto wallets to raise and transfer approximately $200,000 for Hamas-linked operatives, resulting in seizure of the funds.",
 "$200,000 (seized)","Funds Seized; prosecutions ongoing","IEEPA, Anti-Terrorism Act, Bank Secrecy Act",
 "https://www.chainalysis.com/blog/ncet-blanche-memo/",
 "Textbook example of the 'carve-out' the Blanche Memo explicitly preserved: crypto-enabled terrorism financing remains a top DOJ priority despite the registration-enforcement retreat."),

("26-SEC-0021","10-Feb-26","SEC","SEC (Statement on Tokenised Securities Settlement)","Policy Reform",
 "SEC Chairman Paul Atkins announced staff guidance permitting SEC registrants to use GENIUS Act-compliant payment stablecoins for trade settlement and margining, formally bridging traditional securities infrastructure with the new stablecoin regime.",
 "N/A","Staff Guidance","GENIUS Act of 2025; Securities Exchange Act of 1934",
 "https://www.sec.gov/newsroom/speeches-statements/atkins-statement-genius-act-071825",
 "Most recent regulatory development in the dataset; illustrates the shift from enforcement-driven to integration-driven crypto policy that frames the Section 1 and Section 3 discussion."),
]

new_df = pd.DataFrame(new_cases, columns=COLS)
print('New researched records:', len(new_df))
new_df.head(3)


### 1.1.7 Code: Combining, Chronological Sorting, and Field Engineering

The baseline and new-cases DataFrames are merged together, sorted by date, and the three analytical fields outlined in §1.1.4 are created. The date parser is designed to be tolerant of the mixed `DD-Mon-YY` formats that appear in the data, and the fine-amount parser is designed to correctly distinguish and specify if "no fine was published" (`NaN`) from "the action resulted in a fine of exactly zero dollars" (`0.0`, used for case dismissals where a fine was originally sought but ultimately not paid).


In [ ]:

# ---------------------------------------------------------------------------
# Here I combine the refined baseline and newly researched records
# ---------------------------------------------------------------------------
full = pd.concat([base, new_df], ignore_index=True)
print("Combined shape before sorting/engineering:", full.shape)

# Robust date parser tolerant of the dataset's mixed DD-Mon-YY / DD-Mon-YYYY formats.
def parse_date(s):
    s = str(s).strip()
    for fmt in ("%d-%b-%y", "%d-%b-%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(s, format=fmt)
        except Exception:
            continue
    return pd.to_datetime(s, errors="coerce", dayfirst=True)

full["_date_parsed"] = full["Date of Action"].apply(parse_date)
full = full.sort_values("_date_parsed", na_position="last").reset_index(drop=True)

n_unparsed = full["_date_parsed"].isna().sum()
print(f"Rows with unparseable dates after cleaning: {n_unparsed}")


In [ ]:
# Fine-amount parser: this extracts a clean numeric figure (USD millions) from the free-text 'Amount of Fine' column,
# while correctly preserving the distinction between "no monetary fine reported" (NaN) and "an action that
# explicitly resulted in $0 paid" (0.0, e.g. dismissed/closed cases).

def clean_fine(x):
    if not isinstance(x, str):
        return np.nan
    x_clean = x.replace(",", "")

    m = re.search(r"\$\s*([\d.]+)\s*million", x_clean, flags=re.I)
    if m:
        return float(m.group(1))

    m = re.search(r"\$\s*([\d.]+)\s*billion", x_clean, flags=re.I)
    if m:
        return float(m.group(1)) * 1000.0

    # Genuinely non-monetary entries (legislation, pending litigation, sealed
    # settlements, seizures pending forfeiture) -> correctly left missing.
    if re.search(r"n/a|disgorgement and penalties|under seal|pending|seized", x_clean, flags=re.I):
        return np.nan

    if "$0" in x_clean or x_clean.strip() == "$0":
        return 0.0

    m = re.search(r"\$\s*([\d.]+)\b", x_clean)
    if m:
        val = float(m.group(1))
        return val / 1_000_000.0 if val >= 1000 else val

    return np.nan

full["Fine_USD_Millions"] = full["Amount of Fine"].apply(clean_fine)

print(f"Rows with a parsed numeric fine: {full['Fine_USD_Millions'].notna().sum()} / {len(full)}")
print(f"Rows correctly left as non-monetary (NaN): {full['Fine_USD_Millions'].isna().sum()}")


In [ ]:

# ---------------------------------------------------------------------------
# Here, I engineer the Era and Is_Dismissal_or_Closure analytical fields
# ---------------------------------------------------------------------------
INAUGURATION = pd.Timestamp("2025-01-20")
full["Era"] = np.where(
    full["_date_parsed"] < INAUGURATION,
    "Pre-Trump (Biden/Gensler)",
    "Trump Administration",
)

dismissal_keywords = ["dismiss", "closed", "no action", "no enforcement"]
full["Is_Dismissal_or_Closure"] = full["Outcome"].str.lower().apply(
    lambda s: any(k in s for k in dismissal_keywords) if isinstance(s, str) else False
)

full = full.drop(columns=["_date_parsed"])

FINAL_COLS = COLS + ["Fine_USD_Millions", "Era", "Is_Dismissal_or_Closure"]
full = full[FINAL_COLS]

print("Final database shape:", full.shape)
full.to_csv("DataCryptoFines.csv", index=False)
print("Saved refined, expanded database to DataCryptoFines.csv")
full.head()



### 1.1.8 Sanity Checks on the Final Database

The basic relational integrity properties of the constructed database are verified before the EDA can be trusted: no duplicate Case IDs (except for those that are deliberately shared in administrative proceedings), no missing entity names, and a sensible, contiguous date range.


In [ ]:

print("=== Relational integrity checks ===\n")

print("Total rows:", len(full))
print("Date range:", full['Date of Action'].iloc[0], "to", full['Date of Action'].iloc[-1])

dupe_ids = full["Case ID"].value_counts()
dupe_ids = dupe_ids[dupe_ids > 1]
print(f"\nCase IDs shared by more than one row: {len(dupe_ids)}")
if len(dupe_ids) > 0:
    print(full[full['Case ID'].isin(dupe_ids.index)][['Case ID','Entity/Individual','Date of Action']])

print("\nMissing Entity/Individual:", full["Entity/Individual"].isna().sum())
print("Missing Date of Action:", full["Date of Action"].isna().sum())
print("Missing Case ID (expected for policy/legislative events without a docket number):",
      full["Case ID"].isna().sum())

print("\n=== Era split ===")
print(full["Era"].value_counts())

print("\n=== Dismissal/closure rate by era (the key regulatory-easing signal) ===")
print(full.groupby("Era")["Is_Dismissal_or_Closure"].mean().round(3))



---
## 1.2 Exploratory Data Analysis (EDA)

This section performs a full EDA on the cleaned and enriched database, with the categories outlined in the brief (regulatory body versus enforcement category, entity type versus outcome) as well as a temporal dimension necessary to perform the regulatory-easing analysis. I created a transition from univariate summary statistics to missingness/data-shape diagnostics to the bivariate pivot tables and cross-tabulations that make up the backbone of the analysis for this section.

### 1.2.1 Dataset Overview and Descriptive Statistics


In [ ]:

print(full.info())
print("\n--- Numeric summary: Fine_USD_Millions ---")
print(full["Fine_USD_Millions"].describe())


In [ ]:

print("Unique regulatory bodies (raw, before multi-body splitting):", full["Regulatory Body or Bodies"].nunique())
print("Unique entities/individuals:", full["Entity/Individual"].nunique())
print("Unique enforcement categories (raw text):", full["Category of Enforcement"].nunique())
print("Date span:", full["Date of Action"].iloc[0], "->", full["Date of Action"].iloc[-1])



### 1.2.2 Handling Multi-Regulator Cases

Relational-database subtlety in this space: more than one regulator is involved in many of the cryptocurrency enforcement actions (e.g., the Binance resolution was brought by the DOJ, CFTC, and FinCEN). Data on involvement by individual agencies needs to be broken out from the "DOJ, CFTC, FinCEN" category, otherwise it would be an understatement of how frequently each agency gets involved. Therefore, we explode out the `Regulatory Body or Bodies` field into a long format auxiliary table for body-level EDA and keep the original wide-format field for case-level analysis (where each row in `full` must be a single enforcement action, required by the Decision Tree model in §1.3).


In [ ]:

# Long-format explosion: one row per (Case, Regulator) pair, for body-level frequency analysis.
def split_bodies(s):
    if not isinstance(s, str):
        return []
    # Split on commas and the word " and " / " / ", trim whitespace and parenthetical asides.
    parts = re.split(r",|/| and ", s)
    cleaned = []
    for p in parts:
        p = re.sub(r"\(.*?\)", "", p).strip()
        if p:
            cleaned.append(p)
    return cleaned

exploded_rows = []
for idx, row in full.iterrows():
    for body in split_bodies(row["Regulatory Body or Bodies"]):
        exploded_rows.append({"Case_Index": idx, "Regulatory_Body": body, "Era": row["Era"]})

bodies_long = pd.DataFrame(exploded_rows)
print("Exploded (long-format) regulator rows:", len(bodies_long))
print("\nTop 12 regulators by raw frequency of involvement (any case, solo or joint):")
print(bodies_long["Regulatory_Body"].value_counts().head(12))



### 1.2.3 Entity Type Classification

The brief is requesting “entity type vs. outcome” analysis. The raw `Entity/Individual` field contains a combination of named companies, individuals, multi-defendant groups and abstract policy events (e.g. "GENIUS Act (Public Law)").





A straightforward rule-based classifier is used to generate an Entity_Type field with four levels: Corporate Entity, Individual, Multiple Defendants/Joint Respondents, and Policy/Legislative Event. Without this field the requested comparison would be meaningless, since the raw column contains over 100 unique free-text labels that cannot be cross-tabulated directly



In [ ]:

def classify_entity(name, category):
    name_l = str(name).lower()
    cat_l = str(category).lower()

    # Policy / legislative / procedural / regulatory events are identified by EITHER
    # a category keyword OR a parenthetical marker on the entity name itself (these
    # rows represent institutional actions, not named respondents). Checking both
    # fields, rather than category text alone, is necessary because the category
    # labels for these events vary ("Policy Reform", "Procedural Reform", "Regulatory
    # Pause", "Federal Stablecoin Legislation Enacted"): a single keyword match on
    # category text alone misses several of them.
    policy_keywords = ["policy", "reform", "regulatory pause", "legislation enacted",
                        "annual report", "rule amendment"]
    policy_name_markers = ["(policy action", "(public law)", "(rules of practice",
                            "(rule amendment", "(annual report", "(statement"]
    if any(k in cat_l for k in policy_keywords) or any(m in name_l for m in policy_name_markers):
        return "Policy / Legislative Event"

    if any(k in name_l for k in [" inc", " llc", " ltd", " corp", " labs", " group",
                                   " foundation", " trading", " holdings", " network",
                                   " capital", " technologies", " technology", " platform",
                                   "exchange", "company", "co.", "gmbh"]):
        return "Corporate Entity"
    if ";" in name_l or " and " in name_l or "/" in name_l or "three" in name_l or "two " in name_l:
        return "Multiple Defendants / Joint Respondents"
    # Default: a single personal name (e.g. "Roman Storm (Tornado Cash)")
    return "Individual"

full["Entity_Type"] = full.apply(
    lambda r: classify_entity(r["Entity/Individual"], r["Category of Enforcement"]), axis=1
)

print(full["Entity_Type"].value_counts())
print()
print("Sanity check -- all rows now correctly tagged as Policy/Legislative:")
print(full[full["Entity_Type"] == "Policy / Legislative Event"]["Entity/Individual"].tolist())



### 1.2.4 Pivot Tables and Cross-Tabulations

Here it is important that I used advanced pivot tables and "cross tabulations visualizations" to look at relationships, including a regulatory body versus enforcement category, and entity type versus outcome.

Five such tables are built below, accompanied by a visualization, and a short interpretive note. To make a 24-category and mostly-empty cross-tab (the raw `Regulatory Body or Bodies` field has 24 joint-respondent strings) manageable, the long-format `bodies_long` table from §1.2.2 is used for body-level cross-tabs, while the case-level `full` table is used when the unit of analysis needs to be one enforcement action (e.g. outcome or fine-size cross-tab).

#### Pivot Table 1: Regulatory Body × Era (case volume)


In [ ]:

# Merge Era onto the long-format body table for body-level era comparison.
pivot1 = pd.crosstab(bodies_long["Regulatory_Body"], bodies_long["Era"])
pivot1["Total"] = pivot1.sum(axis=1)
pivot1 = pivot1.sort_values("Total", ascending=False)
pivot1


In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))
top_bodies = pivot1.head(8).drop(columns="Total")
top_bodies.plot(kind="barh", stacked=True, ax=ax, color=["#1f77b4", "#d62728"])
ax.set_title("Figure 1.1: Enforcement Involvement by Regulator and Era\n(Top 8 Regulators, Stacked by Pre/Post Inauguration)", fontsize=12, fontweight="bold")
ax.set_xlabel("Number of Cases Involving This Regulator")
ax.set_ylabel("Regulatory Body")
ax.legend(title="Era", loc="lower right")
plt.tight_layout()
plt.savefig("fig1_1_regulator_by_era.png", bbox_inches="tight")
plt.show()



**Interpretation:** The SEC and CFTC are the two agencies that have the largest number of raw cases in the entire period of 2015–2026, the agencies that have primary jurisdiction over securities-classified digital assets and commodities-classified digital assets, respectively.


Given the relatively small time period the Trump-trump slice covers (some 13 months Jan-2025 to Feb-2026), compared to the pre-trump slice (nearly 10 years), it is important to note that comparisons of data in the raw format are not meaningful.


Here, there is not yet any time exposure adjusted. This adjustment is done in a normalisation based on monthly rates in §1.6. This table is used for structural purposes only, to indicate which regulators the rate-based regulatory-easing analysis in this notebook later on should focus on.


#### Pivot Table 2: Category of Enforcement × Era


In [ ]:

# The raw Category of Enforcement field has 80 nearly-unique values (many auto-generated
# during dataset extension to be maximally descriptive). For a meaningful pivot table, I
# first map these onto a small set of families using their respective keyword rules, while preserving the
# original detailed text in the underlying data in order to drill down
def categorise_family(cat):
    c = str(cat).lower()
    if "dismiss" in c or "closed" in c or "no action" in c:
        return "Dismissal / Case Closure"
    if "fraud" in c:
        return "Fraud"
    if "money laundering" in c or "aml" in c or "bank secrecy" in c or "sanctions" in c:
        return "AML / Sanctions / BSA"
    if "unregistered" in c or "securities offering" in c or "broker" in c or "exchange" in c:
        return "Unregistered Securities / Exchange Activity"
    if "policy" in c or "reform" in c or "legislation" in c or "regulatory pause" in c or "annual report" in c:
        return "Policy / Legislative Event"
    if "manipulation" in c or "cherry" in c or "insider" in c:
        return "Market Manipulation / Trading Misconduct"
    return "Other Civil/Administrative Enforcement"

full["Category_Family"] = full["Category of Enforcement"].apply(categorise_family)

pivot2 = pd.crosstab(full["Category_Family"], full["Era"], margins=True, margins_name="Total")
pivot2


In [ ]:

pivot2_pct = pd.crosstab(full["Category_Family"], full["Era"], normalize="columns") * 100
fig, ax = plt.subplots(figsize=(9, 5.5))
pivot2_pct.plot(kind="bar", ax=ax, color=["#1f77b4", "#d62728"])
ax.set_title("Figure 1.2: Enforcement Category Mix by Era\n(% of Cases Within Each Era)", fontsize=12, fontweight="bold")
ax.set_ylabel("% of Cases in Era")
ax.set_xlabel("Enforcement Category Family")
plt.xticks(rotation=35, ha="right")
ax.legend(title="Era")
plt.tight_layout()
plt.savefig("fig1_2_category_by_era_pct.png", bbox_inches="tight")
plt.show()



**Interpretation:** Normalising (to percentage within Era) is important here as the two Eras have very different sample sizes (110 vs 32 cases). The percentage view makes it clear there has been a shift in composition. “Dismissal / Case Closure” and “Policy / Legislative Event” accounts for a much bigger portion of the Trump-era caseload than the pre-Trump caseload, while “Unregistered Securities / Exchange Activity”, the SEC's own “regulation-by-enforcement” theory, shrinks as a percentage of the total.


Both eras continue to have the categories of "Fraud" and "AML / Sanctions / BSA," which is in keeping with Blanche Memo's disclosures that these categories will not be deprioritised, but rather continue to be handled with priority, even if registration cases are not prioritized.


#### Pivot Table 3: Regulatory Body × Enforcement Category Family (Cross-Tabulation)


In [ ]:

body_cat = bodies_long.merge(
    full[["Category_Family"]], left_on="Case_Index", right_index=True, how="left"
)
pivot3 = pd.crosstab(body_cat["Regulatory_Body"], body_cat["Category_Family"])
pivot3 = pivot3.loc[pivot3.sum(axis=1).sort_values(ascending=False).head(6).index]  # top 6 regulators
pivot3


In [ ]:

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot3, annot=True, fmt="d", cmap="Blues", linewidths=0.5, ax=ax, cbar_kws={"label": "Number of Cases"})
ax.set_title("Figure 1.3: Regulator × Enforcement Category Cross-Tabulation\n(Top 6 Regulators by Total Involvement)", fontsize=12, fontweight="bold")
ax.set_xlabel("Enforcement Category Family")
ax.set_ylabel("Regulatory Body")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("fig1_3_regulator_category_heatmap.png", bbox_inches="tight")
plt.show()



**Interpretation:** This cross tabulation reinforces the expected jurisdictions for the agencies: SEC focuses on Unregistered Securities/Exchange Activity and Fraud, the CFTC and DOJ focus on Fraud and AML/Sanctions/BSA, and Policy/Legislative Events are spread across SEC, CFTC, DOJ and FinCEN, consistent with the coordinated, multi-agency nature of the 2025 enforcement-policy reversal described in §1.6.

#### Pivot Table 4: Entity Type × Outcome (Cross-Tabulation)


In [ ]:

def outcome_family(o):
    o_l = str(o).lower()
    if "dismiss" in o_l or "closed" in o_l or "no enforcement" in o_l or "no action" in o_l:
        return "Dismissed / Closed (No Penalty)"
    if "convict" in o_l or "guilty" in o_l or "sentenced" in o_l or "default judgment" in o_l:
        return "Convicted / Default Judgment"
    if "settle" in o_l or "consent" in o_l:
        return "Settled / Consent Order"
    if "charge" in o_l and "pending" in o_l or "indicted" in o_l or "ongoing" in o_l:
        return "Charged / Litigation Ongoing"
    if "policy" in o_l or "guidance" in o_l or "rule" in o_l or "report" in o_l or "enacted" in o_l or "announce" in o_l:
        return "Policy / Procedural Outcome"
    return "Other / Final Judgment"

full["Outcome_Family"] = full["Outcome"].apply(outcome_family)

pivot4 = pd.crosstab(full["Entity_Type"], full["Outcome_Family"], margins=True, margins_name="Total")
pivot4


In [ ]:

pivot4_viz = pd.crosstab(full["Entity_Type"], full["Outcome_Family"])
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot4_viz, annot=True, fmt="d", cmap="OrRd", linewidths=0.5, ax=ax, cbar_kws={"label": "Number of Cases"})
ax.set_title("Figure 1.4: Entity Type × Outcome Cross-Tabulation", fontsize=12, fontweight="bold")
ax.set_xlabel("Outcome Family")
ax.set_ylabel("Entity Type")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("fig1_4_entitytype_outcome_heatmap.png", bbox_inches="tight")
plt.show()



**Interpretation:** Corporate entities make up the majority of “Settled/Consent Order” outcomes (the SEC and CFTC's typical enforcement approach against exchanges and platforms) and “Individuals” are overrepresented among the “Convicted/Default Judgment” outcomes, which aligns with the Blanche Memo's informational memo that prioritizes enforcement against natural persons who defrauded investors over platforms that facilitated the fraud.


The clustering of the events under the Policy/Legislative events type is consistent with the classification logic in §1.2.3 and the outcome data in that the events cluster entirely under Policy/Procedural Outcome.

#### Pivot Table 5: Median and Total Fine Size by Era and Category Family (Advanced Pivot with Aggregation)


In [ ]:

pivot5 = full.pivot_table(
    values="Fine_USD_Millions",
    index="Category_Family",
    columns="Era",
    aggfunc=["median", "sum", "count"],
)
pivot5


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

median_fine = full.pivot_table(values="Fine_USD_Millions", index="Category_Family", columns="Era", aggfunc="median")
median_fine.plot(kind="barh", ax=axes[0], color=["#1f77b4", "#d62728"], legend=False)
axes[0].set_title("Median Fine (USD Millions) by Category & Era", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Median Fine Among Cases WITH a Published Fine (log scale)")
axes[0].set_xscale("symlog")

total_fine = full.pivot_table(values="Fine_USD_Millions", index="Category_Family", columns="Era", aggfunc="sum")
total_fine.plot(kind="barh", ax=axes[1], color=["#1f77b4", "#d62728"], legend=False)
axes[1].set_title("Total Fines (USD Millions) by Category & Era", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Total Fines (USD Millions, log scale)")
axes[1].set_xscale("symlog")

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, title="Era", loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2)

plt.suptitle("Figure 1.5: Fine Size by Enforcement Category and Era", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig1_5_finesize_category_era.png", bbox_inches="tight")
plt.show()


An important qualification on interpreting the median panel correctly is that the apparent increase in the median fine for the violation of “Unregistered Securities / Exchange Activity” from pre-Trump era (USD 1.5m) to Trump-era (USD 2.1m) is not evidence that such “registration” cases became more expensive during the new administration, it is a survivorship artefact.


This column gives the number of cases in that category for which the value for the variable is (In addition, Coinbase's fine was the only one that did not occur in the Trump era, which is why the median number of cases in any publication dropped from 13 (pre-Trump) to 4 (Trump era); the rest of that drop is accounted for by the fact that a large number of the Trump-era registration-theory cases were dismissed with no fine at all and therefore are correctly omitted from the median number of fines calculation (per the missingness logic in §1.1.7).)


The median for the four that did still have a fine, just represents the particular cases that were left to resolution but does not mean the policy changed towards a bigger fine.


The right summary statistic here is not the median size of the rare final fines, it's the number of cases with a fine, which dropped by more than half.


The right hand Total Fines panel is a much more accurate reflection of the easing as it reflects the combined impact of fewer fine cases and less cumulative fines in all categories other than Fraud/ AML/Sanctions/BSA.


#### Pivot Table 6: Count of Fine-Bearing vs. Fine-Free Cases by Era (resolving the survivorship issue directly)


In [ ]:

full["Has_Published_Fine"] = full["Fine_USD_Millions"].notna() & (full["Fine_USD_Millions"] > 0)

pivot6 = pd.crosstab(full["Era"], full["Has_Published_Fine"])
pivot6.columns = ["No Fine / Dismissed / Non-Monetary", "Fine > $0 Published"]
pivot6["% With a Fine"] = (100 * pivot6["Fine > $0 Published"] /
                            (pivot6["Fine > $0 Published"] + pivot6["No Fine / Dismissed / Non-Monetary"])).round(1)
pivot6


In [ ]:

fig, ax = plt.subplots(figsize=(7, 5))
pivot6["% With a Fine"].plot(kind="bar", ax=ax, color=["#1f77b4", "#d62728"])
ax.set_title("Figure 1.6: Share of Cases Resulting in an Actual Published Fine,\nby Era", fontsize=12, fontweight="bold")
ax.set_ylabel("% of Cases With a Fine > $0")
ax.set_xlabel("")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for i, v in enumerate(pivot6["% With a Fine"]):
    ax.text(i, v + 1.5, f"{v}%", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("fig1_6_pct_with_fine_by_era.png", bbox_inches="tight")
plt.show()



**Interpretation:** This single chart is the most "clean" summary statistic for the entire EDA on the central question of the brief.

The percentage of enforcement-related database entries to those that led to a public monetary penalty dropped significantly from one period (direct) to another (indirect).

Clearer evidence of the enforcement-easing pattern than the median-fine comparison above, the latter of which is marred by a survivorship bias.




### 1.2.5 Temporal Trend Analysis: Enforcement Volume and Dismissal Rate Over Time

None of the above cross-section pivot tables capture the particularity of the change in enforcement intensity over the course of the decade, or the exact date of the change, when the Trump administration took office. The real date field is now rebuilt (parsed once and kept here to be used for time-series) and then quarterly bins are created to view the trend directly.


Most directly answering the brief's command to "examine and visualize how enforcement actions by regulators have eased since the inauguration of President Trump," is this chart.



In [ ]:

full["_date"] = full["Date of Action"].apply(parse_date)
full["Quarter"] = full["_date"].dt.to_period("Q")

quarterly = full.groupby("Quarter").agg(
    Total_Cases=("Case ID", "size"),
    Dismissals=("Is_Dismissal_or_Closure", "sum"),
    Total_Fines_USDm=("Fine_USD_Millions", "sum"),
).reset_index()
quarterly["Quarter_str"] = quarterly["Quarter"].astype(str)
quarterly["Dismissal_Rate_Pct"] = (100 * quarterly["Dismissals"] / quarterly["Total_Cases"]).round(1)

# Restrict the plotted window to 2022 Q1 onward for readability; the database's earliest
# entries (2015-2021) are sparse (1-2 cases per year) and would compress the recent,
# analytically critical quarters into an unreadable sliver of the x-axis.
quarterly_recent = quarterly[quarterly["Quarter"] >= pd.Period("2022Q1", freq="Q")].copy()
quarterly_recent


In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

axes[0].bar(quarterly_recent["Quarter_str"], quarterly_recent["Total_Cases"],
            color=["#d62728" if q >= pd.Period("2025Q1", freq="Q") else "#1f77b4"
                   for q in quarterly_recent["Quarter"]])
axes[0].set_title("Figure 1.7: Quarterly Enforcement-Related Case Volume (2022 Q1 – 2026 Q1)\n"
                   "Red = Post-Inauguration Quarters", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Number of Cases / Events")
axes[0].axvline(x="2025Q1", color="black", linestyle="--", linewidth=1.2)
quarterly_recent_reset = quarterly_recent.reset_index(drop=True)
inaug_pos = quarterly_recent_reset.index[quarterly_recent_reset["Quarter_str"] == "2025Q1"]
if len(inaug_pos) > 0:
    axes[0].annotate("Inauguration\n(20-Jan-2025)", xy=(inaug_pos[0], axes[0].get_ylim()[1]*0.92),
                      xytext=(inaug_pos[0] + 0.4, axes[0].get_ylim()[1]*0.92),
                      fontsize=9, style="italic", va="top")

axes[1].plot(quarterly_recent["Quarter_str"], quarterly_recent["Dismissal_Rate_Pct"],
             marker="o", color="#9467bd", linewidth=2)
axes[1].set_title("Quarterly Dismissal / Closure Rate (%)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("% of Quarter's Cases Dismissed or Closed")
axes[1].set_xlabel("Quarter")
axes[1].axvline(x="2025Q1", color="black", linestyle="--", linewidth=1.2)

plt.setp(axes[1].get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.savefig("fig1_7_quarterly_trend.png", bbox_inches="tight")
plt.show()



The dismissal-rate panel suggests that it was a step-change, but centered almost entirely in the quarters following the January 2025 inauguration (2025 Q1–Q2), corresponding to the timeline of events: the SEC Crypto Task Force was announced on 23-Jan-25; the dismissal of Coinbase occurred on 27-Feb-25; and the National Cryptocurrency Enforcement Team was disbanded by the DOJ on 7-Apr-25.


This is indicative of a true, datable policy change, and not simply a secular trend which coincided with the change of administration, which is important for the regulatory discussion in §1.6.

### 1.2.6 Correlation Structure Among Numeric/Engineered Features

Finally, before jumping into predictive modelling, a quick correlation check of the numeric and boolean fields used in a Decision Tree confirms that there's no obvious problem redundancy in engineered features (Is_Dismissal_or_Closure should not be 100% predictable from Era alone, for example because a number of non-Trump cases were also dismissed/closed etc.).


In [ ]:

full["Era_Binary"] = (full["Era"] == "Trump Administration").astype(int)
full["Has_Fine_Binary"] = full["Has_Published_Fine"].astype(int)
full["Is_Dismissal_Binary"] = full["Is_Dismissal_or_Closure"].astype(int)

corr_cols = ["Fine_USD_Millions", "Era_Binary", "Has_Fine_Binary", "Is_Dismissal_Binary"]
corr_matrix = full[corr_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Figure 1.8: Correlation Matrix\n(Numeric & Engineered Boolean Features)", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("fig1_8_correlation_matrix.png", bbox_inches="tight")
plt.show()



**Interpretation:** Well, numerically, it can be seen that this correlation is not 100% (not ±1) but, that `Era_Binary` correlates moderately (not perfectly) with both `Is_Dismissal_Binary` (positive) and `Has_Fine_Binary` (negative), confirming what is shown visually in Figures 1.6 and 1.7 is informative and not a repetition of the outcome variables.

This is relevant to the Decision Tree model in §1.3: It shows that the era features are indeed predictive of `Fine_USD_Millions` and not just built directly into the target.




---
## 1.3 Predictive Modelling: A Decision Tree for Fine Amount
Using this data, a decision tree was built to predict the amount of fines.

The task required in the brief is to create a Decision Tree model to predict the amount of the crypto fine based on the features in the database, and explaining feature-selection, the training of the model, and evaluation of the accuracy of the model. This is not a classification problem because the wording of the brief, "predict the amount", suggests that this is a regression problem, where a continuous value (here the value in millions of USD) is estimated. So the `DecisionTreeRegressor` is the family of models that is directly applicable to this situation, for a data set of this size. A single decision tree is known to be high-variance, and the question was also to perform an evaluation of the accuracy of the model, which is more useful when compared against another.

### 1.3.1 Feature Selection Rationale

The training or evaluation of a fine-amount regressor should only be done on the cases that have a fine amount posted and not missing. (The dismissed cases, the pending-litigation cases and the policy/legislative cases don't have a fine amount to be predicted, and so the latter would only teach the model the wrong lesson: that "dismissal" is the same as "a $0 settlement", which is analytically incorrect, as established in §1.1.4 and §1.2.5). This limits the modelling sample to the rows where Has_Published_Fine is informative about the sub-question (how large was the fine, given that one was levied) which is often the most common one.
and that real world examples of such scenarios are realistic and precedent only ever related to fine bearing ones (e.g., a regulator's own internal severity-prediction method would only be trained on already-resolved, fine-bearing precedent).

The candidate features (and the rationale for including/excluding each feature) are:


| Feature | Included? | Rationale |
|---|---|---|
| `Regulatory Body or Bodies` (categorical) | **Yes** | Different regulators have systematically different fine scales (e.g. SEC settlements with large exchanges vs. CFTC actions against individual fraudsters) — directly relevant to predicting severity. |
| `Category_Family` (categorical, engineered in §1.2.4) | **Yes** | Captures the *type* of violation, which the EDA in Figure 1.5 already showed is strongly associated with fine magnitude (AML/Sanctions cases differ systematically from Fraud cases, for instance). |
| `Entity_Type` (categorical, engineered in §1.2.3) | **Yes** | Corporate entities and multi-defendant groups tend to attract larger aggregate penalties than individuals — a structural, not coincidental, pattern. |
| `Era` (categorical, engineered in §1.1.4) | **Yes** | Directly operationalises the brief's central regulatory-easing question: does the *same type* of case, holding category and entity type constant, now attract a different fine under the Trump administration? Including it lets the tree's feature-importance ranking answer this quantitatively rather than just visually. |
| `Outcome_Family` (categorical, engineered in §1.2.4) | **No** | Excluded as a *target leak*: `Outcome_Family` is partly definitionally entangled with the existence of a fine (e.g. "Dismissed" rows have no fine by construction), so including it would let the tree trivially "predict" the target from its own definition rather than learning a genuine relationship. |
| `Is_Dismissal_or_Closure` | **No** | Same target-leakage concern as above — by definition, every row used for training already has `Is_Dismissal_or_Closure == False`, so this feature would be a constant within the training sample and carries zero information once the fine-bearing subset is selected. |
| `Date of Action` / raw text fields (`Description of Violation`, `Additional Notes`, etc.) | **No** | Free text is out of scope for a Decision Tree without a separate NLP feature-extraction pipeline (TF-IDF, embeddings), which is not what the brief is asking for in Section 1; the *Era* categorical already captures the temporal signal that matters for this assessment's purpose. |

### 1.3.2 Data Preparation


In [ ]:

model_df = full[full["Has_Published_Fine"]].copy()
print(f"Rows available for fine-amount modelling (fine > $0 published): {len(model_df)}")

FEATURES = ["Regulatory Body or Bodies", "Category_Family", "Entity_Type", "Era"]
TARGET = "Fine_USD_Millions"

# The raw 'Regulatory Body or Bodies' field has many low-frequency joint-respondent strings
# (e.g. "DOJ, FBI, German Federal Criminal Police (Bundeskriminalamt)") which would each become
# their own one-hot column and overfit on a sample this size. We collapse any body string that
# is not one of the five most frequent *primary* regulators into an explicit "Other/Joint" bucket,
# preserving genuine signal while controlling dimensionality
# for small-sample categorical encoding.
top_bodies = bodies_long["Regulatory_Body"].value_counts().head(5).index.tolist()
print("Top 5 individual regulators retained as distinct categories:", top_bodies)

def simplify_body(s):
    s = str(s)
    if s in top_bodies:
        return s
    # if the joint string's FIRST listed regulator is one of the top 5, keep that as a proxy
    first = re.split(r",|/", s)[0].strip()
    if first in top_bodies:
        return first
    return "Other / Joint Regulator"

model_df["Regulatory_Body_Simplified"] = model_df["Regulatory Body or Bodies"].apply(simplify_body)
print(model_df["Regulatory_Body_Simplified"].value_counts())


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

MODEL_FEATURES = ["Regulatory_Body_Simplified", "Category_Family", "Entity_Type", "Era"]
X = model_df[MODEL_FEATURES]
y = model_df[TARGET]

# A log1p transform of the (heavily right-skewed, multi-billion-dollar-range) target is applied
# before training: fine sizes span from $0.06m to several thousand $m, and an untransformed
# Decision Tree would have its splits dominated entirely by a handful of outlier mega-settlements
# (Binance $4.31bn, Celsius $4.7bn). log1p compresses this range, lets the tree learn structure
# across the *whole* distribution rather than just the extreme tail, and is inverted (np.expm1)
# before any metric is reported, so all evaluation figures below are in genuine USD millions.
y_log = np.log1p(y)

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.25, random_state=RANDOM_STATE
)
print(f"Training rows: {len(X_train)}  |  Test rows: {len(X_test)}")



### 1.3.3 Encoding Pipeline and Model Training

To train a model, the first step is to encode the pipeline.The first step in training a model is to encode the pipeline.

All four model features are categorical, and a `OneHotEncoder` is used in a `ColumnTransformer` / `Pipeline`, ensuring that preprocessing and the estimator are kept together as one reproducible object (without falling into the all-too-prevalent reproducibility pitfall of fitting one encoder and then silently using stale encoder state in future runs).


**Hyperparameter Rationale:** The number of training rows is only 66, and there are only four categorical columns (one-hot expanded to ~15-20), so it is intentionally not allowed to grow too deep with the `max_depth` argument, and `min_samples_leaf` is set above the scikit-learn default of 1 to prevent the tree from splitting down to a single leaf for each training row, so as to capture true majority patterns. These values were selected using a small manually inspected grid (depths 2–6, min leaf sizes 3–10) only by the cross-validated performance on the training set; the held-out test set is only used once, at the end, to evaluate the final performance, avoiding leakage of the test set in model selection.


In [ ]:

from sklearn.model_selection import GridSearchCV, KFold

preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), MODEL_FEATURES)]
)

dt_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", DecisionTreeRegressor(random_state=RANDOM_STATE)),
])

# Small, defensible grid given the sample size; selection is via 5-fold CV on the training
# set ONLY, so the test set remains a genuinely held-out evaluation set.
param_grid = {
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__min_samples_leaf": [3, 4, 5, 7, 10],
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid_search = GridSearchCV(
    dt_pipeline, param_grid, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1
)
grid_search.fit(X_train, y_train_log)

print("Best hyperparameters (selected via 5-fold CV, training set only):")
print(grid_search.best_params_)
print(f"Best CV MAE (log-fine scale): {-grid_search.best_score_:.4f}")

best_dt = grid_search.best_estimator_



### 1.3.4 Held-Out Test Set Evaluation

The Held-Out Test Set Evaluation will evaluate the responses of the test set that is not included in the training.

The tuned tree is now tested for the first time on the test set which it has not seen during training nor hyperparameter tuning. Each number reported below is directly interpretable in real fine-size terms because any value is predicted as if it were a USD-million (intake) and then transformed back to a USD-million (intake) by applying the inverse of the exponential transformation (`np.expm1`).



In [ ]:

y_pred_log = best_dt.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test = np.expm1(y_test_log)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# R^2 and RMSE on the raw-dollar scale are both heavily influenced by one or two
# multi-billion-dollar outlier settlements in the test fold; the log-scale metrics are reported
# alongside as a fairer, outlier-robust view of how well the model orders/ranks fine severity.
mae_log = mean_absolute_error(y_test_log, y_pred_log)
r2_log = r2_score(y_test_log, y_pred_log)

print("=== Decision Tree Regressor: Held-Out Test Set Performance ===\n")
print(f"Test rows: {len(y_test)}\n")
print(f"MAE  (USD millions, raw scale):  {mae:,.2f}")
print(f"RMSE (USD millions, raw scale):  {rmse:,.2f}")
print(f"R^2  (raw scale):                {r2:.3f}\n")
print(f"MAE  (log1p scale):              {mae_log:.3f}")
print(f"R^2  (log1p scale):               {r2_log:.3f}")


In [ ]:

results_df = pd.DataFrame({
    "Actual_Fine_USDm": y_test.values,
    "Predicted_Fine_USDm": y_pred,
    "Era": X_test["Era"].values,
    "Category": X_test["Category_Family"].values,
}).sort_values("Actual_Fine_USDm", ascending=False)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
results_df


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].scatter(y_test, y_pred, alpha=0.7, s=60, c="#1f77b4", edgecolor="k", linewidth=0.4)
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([0, max_val], [0, max_val], "r--", linewidth=1.3, label="Perfect Prediction")
axes[0].set_xlabel("Actual Fine (USD Millions)")
axes[0].set_ylabel("Predicted Fine (USD Millions)")
axes[0].set_title("Actual vs. Predicted Fine\n(Raw Scale)", fontsize=11, fontweight="bold")
axes[0].set_xscale("symlog")
axes[0].set_yscale("symlog")
axes[0].legend()

axes[1].scatter(y_test_log, y_pred_log, alpha=0.7, s=60, c="#2ca02c", edgecolor="k", linewidth=0.4)
max_log = max(y_test_log.max(), y_pred_log.max())
axes[1].plot([0, max_log], [0, max_log], "r--", linewidth=1.3, label="Perfect Prediction")
axes[1].set_xlabel("Actual log(1 + Fine)")
axes[1].set_ylabel("Predicted log(1 + Fine)")
axes[1].set_title("Actual vs. Predicted Fine\n(log1p Scale = Used for Model Selection)", fontsize=11, fontweight="bold")
axes[1].legend()

plt.suptitle("Figure 1.9: Decision Tree: Held-Out Test Set Fit Quality", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig1_9_dt_actual_vs_predicted.png", bbox_inches="tight")
plt.show()



### 1.3.5 Interpreting the Test-Set Result Honestly

Interpretation of the Test-Set Result Honestly:
If the test-set result is not 1,1,1, then the device is most likely damaged.

The above R² is negative, (−0.03 on the raw scale, −0.17 on the log scale), indicating that by this measure, the predictions of the tuned tree are not better than the predictions of always returning the mean of the training set fine. This is not necessarily adjusted or hidden away, and once the cause of it is known, it can be analytically informative, as is the case here:

1. **Extreme outlier concentration.** The $4.7 billion Celsius case is a single test-fold case that is about 19× bigger than the next largest case in the entire dataset. This one row dominates the RMSE (which squares errors) and even if the model performs fairly well on all the other cases, this one outlier will result in a poor RMSE/R2 when it is in the test split.

2. **Wide within-category dispersion.** The "Other Civil/Administrative Enforcement" bucket covers a range of fines from $0.01m to $4,700m or five orders of magnitude.  If two settlements have the same (Regulator × Category × Entity Type × Era) profile, then a four-feature Decision Tree, whose leaves are limited to returning one fixed constant per profile, can not tell the difference between a $10,000 and a $4.7 billion settlement. This is a real feature-set limitation, not a tuning problem: no further grid search over the other parameters (max_depth, min_samples_leaf, etc.) will be able to resolve this issue if the available features are unable to distinguish within-category outliers. These are not numeric features and are not included here on purpose; a production grade severity model would have such features (e.g. assets under management, victim exposure from a user fund).


The held-out variance (or R²) is high-variance when the number of test rows is small, regardless of model quality (as the single mispredicted outlier can drive the value up and down a lot).


The model's "ability to make point predictions" (examined, for instance, through the tree structure below) is not something that should be relied upon for any particular case, and that is exactly the right take-away from the brief's instruction to "evaluate the model's accuracy.What the model still can do credibly, and what it cannot: the tree's "ranking" of typical case severity by category and regulator, examined for instance through the feature importance below, remains interpretable and broadly consistent with the EDA in §1.2.



### 1.3.6 Robustness Benchmark: Random Forest Comparator

Random Forest Comparator is a robustness benchmark.Random Forest Comparator is a benchmark for robustness with the technique of random forest.

A small Random Forest (an ensemble of trees, each of which sees a Bootstrap resample of the data and a random selection of the features) is fit using the same features (and the same train/test split), just to see if the poor R² of the single tree is a single-tree artefact, or if there is a more fundamental limitation in the features used.



In [ ]:

rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200, max_depth=4, min_samples_leaf=5,
        random_state=RANDOM_STATE
    )),
])
rf_pipeline.fit(X_train, y_train_log)

rf_pred_log = rf_pipeline.predict(X_test)
rf_pred = np.expm1(rf_pred_log)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)
rf_r2_log = r2_score(y_test_log, rf_pred_log)

comparison = pd.DataFrame({
    "Model": ["Decision Tree (tuned)", "Random Forest (200 trees, benchmark)"],
    "MAE (USD millions)": [mae, rf_mae],
    "RMSE (USD millions)": [rmse, rf_rmse],
    "R^2 (raw scale)": [r2, rf_r2],
    "R^2 (log1p scale)": [r2_log, rf_r2_log],
})
comparison



**Interpretation:** The diagnosis in §1.3.5 is confirmed by the Random Forest's metrics, which are in the same poor range as the single tree's (both still negative / near-zero R²), which indicate that this is a feature-set ceiling, not a single-tree weakness that ensembling could fix.

An average of 200 trees based on the same four categorical features cannot produce information about severity of the situation (assets under management, number of victims, negotiation dynamics for settlement) that was not already included in the model. That is actually an answer to the brief's "evaluate the model's accuracy" requirement, which is just as well that it is an honest answer, because the conclusion here is that the data on location and type of violation is sufficient to know about the tendency to issue larger or smaller fine amounts, but not sufficient to know precisely how much.


It is important to consider how to gauge the significance of the features, as well as the structure of the trees.


### 1.3.7 Feature Importance and Tree Structure


In [ ]:

# Recover human-readable feature names after one-hot encoding for the importance plot.
ohe = best_dt.named_steps["preprocess"].named_transformers_["cat"]
feature_names = ohe.get_feature_names_out(MODEL_FEATURES)
importances = best_dt.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df = importance_df[importance_df["Importance"] > 0].sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(importance_df))))
ax.barh(importance_df["Feature"], importance_df["Importance"], color="#1f77b4")
ax.set_title("Figure 1.10: Decision Tree Feature Importance\n(One-Hot Encoded Features With Non-Zero Importance)", fontsize=12, fontweight="bold")
ax.set_xlabel("Gini Importance")
plt.tight_layout()
plt.savefig("fig1_10_dt_feature_importance.png", bbox_inches="tight")
plt.show()



**A further, analytically important observation:** Analytically important observation #2: The feature importance chart doesn't have any `Era` in it. The tuned tree did not find any split using `Era` worth using. It should not be read as a rebuttal to the so-called 're-regulatory' story that has been built up


The terms are not used consistently throughout this notebook, but rather represent the more specific and narrow question being posed by this model.



This Decision Tree is only trained on the 89 cases for which a published fine (not a dismissal or closure) exists, and so excludes dismissals and closures (§1.3.1) by design. This regulatory easing does not really happen through systematically lowering the fine amount for the few cases that still are fined, as reflected in Figure 1.6 (72.7% → 28.1%), but through whether or not the case ultimately results in a fine at all, and through which category of case is even being brought, as reflected in Figure 1.2. That is, in the easing, we do not see it in the arithmetic of finding monetary resolution to cases that survive, but we see the easing in case selection and case dismissal.


This is indeed a negative result and it would be a welcome clarification of the regulatory debate in §1.6 rather than a detraction.



In [ ]:

fig, ax = plt.subplots(figsize=(20, 9))
plot_tree(
    best_dt.named_steps["model"],
    feature_names=feature_names,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
    proportion=False,
    precision=2,
)
ax.set_title("Figure 1.11: Fitted Decision Tree Structure\n(Leaf Values Are log1p(Fine in USD Millions); Training Sample, Depth Capped at 3)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig1_11_dt_structure.png", bbox_inches="tight", dpi=140)
plt.show()



---
## 1.4 Regulatory Discussion: The Easing of Enforcement and the New Stablecoin Paradigm

### 1.4.1 What the Data Show About the Regulatory Stance Shift

Regulatory Discussion explores how the enforcement landscape is changing and the emergence of a new stablecoin paradigm.

This refers to the regulatory stance shift, specifically the data and what they indicate.

Based on the EDA in §1.2, and the modelling result in §1.3, four distinct findings that are each defensible in their own right arise regarding the change in the U.S. crypto regulatory stance since 20 Jan. 2025:

This dismissal-rate series in Figure 1.7 indicates a true policy change, not a progressive trend: The SEC's Crypto Task Force (23-Jan-25, two days after the inauguration); the Coinbase dismissal (27-Feb-25); and the DOJ's Blanche Memo, which dismantled the National Cryptocurrency Enforcement Team (7-Apr-25). This would counter the other interpretation, that the data is merely reflecting a gradual secular downward swing in crypto enforcement activities which coincided with a change of administration.

The single most obvious statistic in this analysis is Figure 1.6, which shows the share of rows in the database of enforcement activities that also lead to an actual published fine dropped from 72.7% (pre-Trump) to 28.1% (Trump administration).

The same thing is now confirmed model-based, from the other direction: The Decision Tree shows no useful split on `Era` (§1.3.7) whatsoever, which is the other side of the same observation; for those that did result in a fine, the typical size of the fine is not detectably different by era.


After controlling for the effects of category and regulator. Two findings dovetail with each other: not only were the two regulatory agencies, SEC and CFTC, and the enforcement agency, DOJ, not quietly negotiating smaller settlements for the same conduct, but they were also not even bringing or pursuing the case at all.

The Trump administration's SEC and CFTC and DOJ were not bringing or pursuing the case at all, especially in the case of Coinbase, and especially in the case of the SEC civil claims against Binance, and especially in the case of Kraken, and especially in the case of the investigation at Gemini, Uniswap, OpenSea, Ian Balina and Pinnacle Advisors are the ones that are documented in the database.


Several of the 2025–2026 rows in the constructed database are not actually easing but rather are dismissals: the SafeMoon/Karony fraud conviction, the Unicoin fraud charges, the FTX/Alameda final consent judgments, the $338.7 million CFTC default judgment against a foreign binary-options network, and critically for the “is this just deregulation” reading, continuing, undiminished enforcement against national-security-adjacent conduct (the Tornado Cash/Roman Storm prosecution, the Hamas-financing disruption, and a new iteration of Garantex, which OFAC has succeeded in designating). These categories by name are explicitly preserved by the Blanche Memo. The one directly attested by the database, is not total withdrawal, it is rather redirection, away from registration-theory civil enforcement against legitimate-looking platforms to positive enforcement of fraud and sanctions/national-security issues for bad-faith players.

(iv) Regulators' own published numbers bear this out. Indeed, the SEC's own FY2025 Enforcement Year-in-Review [2] is in the database (Case ID `26-SEC-0008`), and the report shows that the SEC is enforcing fewer cases (across the board) than in the last ten years (313 new actions, the lowest total in a decade, and 27% fewer cases than in FY2024) and that the monetary settlements are less (across the board) than in the last ten years ($808 million total, 45% fewer than FY2024), confirming that the results in this notebook are not an artefact of the way this particular data is compiled.

The GENIUS Act is not only less enforcement, but is a new paradigm.

The brief invites us to reflect on how the new legislation for stablecoins will make it a new paradigm for digital assets, different from the regulatory paradigm of stablecoins that came before it. The pivot point of this argument is the "25-GENIUS-0001" line in the dataset (the signing of the Act, 18-Jul-25), and the change is true change in structure, not just a change in enforcement vigor:


| Dimension | Pre-GENIUS Act Approach (2021–2024) | Post-GENIUS Act Approach (from Jul-2025) |
|---|---|---|
| **Source of rules** | Case-by-case enforcement actions and litigated court rulings (the SEC's Howey-test theory applied retrospectively, often *after* a platform had already operated for years) | A single, prospective federal statute defining a registration regime *before* issuers operate |
| **Legal status of the asset class** | Contested — the SEC argued many tokens (including, at times, payment-type stablecoins) could be unregistered securities | Explicitly settled by statute: GENIUS Act-compliant payment stablecoins are **neither securities nor commodities**, divesting SEC/CFTC jurisdiction over them entirely |
| **Collateral standard** | No federal statutory requirement; varied by issuer (TerraUSD's 2022 collapse, which is *not* in this 1:1-collateral category, exemplified the risk of inadequately backed designs) | Mandatory 100% reserve backing in cash and short-term Treasurys, monthly public disclosure of reserve composition |
| **AML/sanctions treatment** | Inconsistent; depended on whether FinCEN classified a given issuer as a money-services business | Explicit: every permitted payment stablecoin issuer is statutorily a "financial institution" under the Bank Secrecy Act, with mandatory KYC/sanctions-screening programmes |
| **Insolvency treatment** | Untested/uncertain; stablecoin holders had no guaranteed priority in a bankruptcy | Statutory priority claim for stablecoin holders over reserve assets, ahead of other creditors, plus exclusion of reserves from the bankruptcy estate |
| **Geographic reach** | U.S. enforcement reached domestic platforms; foreign issuers operated largely outside U.S. jurisdiction unless individually pursued | Foreign issuers may access the U.S. market only via an explicit comparable-regulator and OCC-registration pathway, with a Treasury-administered non-compliance designation process |

This is the more important, more profound point that the brief is trying to elicit: the GENIUS Act does not merely reduce the regulatory burden on stablecoin issuers from that of the previous enforcement-driven regime, but rather increases the regulatory burden on stablecoin issuers that comply with the Act (mandatory reserves, mandatory disclosure, statutory BSA status, statutory insolvency priority) while also eliminating the prior, contested securities-law overhang that made the status of the asset class itself uncertain. The paradigm shift is, from litigation-defined boundaries to legislature-defined boundaries, which is exactly what industry critics of “regulation by enforcement” have been complaining about for years, and exactly what led the SEC to create its own Crypto Task Force (Case ID `25-SEC-0019`). This new collateral/disclosure regime has balance-sheet and Merton model implications which are developed in full theoretical depth in Section 3 of this assessment.


### 1.4.3 Limitations of This Analysis
This analysis was limited in several ways:This analysis was limited in a number of ways:

As in §1.3.5, it is important to be forthright in noting that this analysis of Section 1 has three limitations:


1. Sample size and asymmetry of the time-window. This dataset has a much narrower window for the Trump-administration era, covering about thirteen months, as compared to nearly a decade before Trump; while §1.2.5 points toward taking a quarterly granularity, a conclusion about the durability of the new equilibrium (vs. an initial dismissals spike in a transition period) would be better served by another 12–24 months of data.

2. Selection of which case(s) to add by the researcher in 2025-2026. While each addition is sourced and can be verified, the selection of cases to feature first was made by the researcher (not a systematic, comprehensive search of the docket of every regulator), which may introduce some selection bias in favor of more publicized cases, which received law firm memos, over less publicized regulatory actions.

3. This notebook’s predictive model is not intended to be used as a deployable system to predict severity, but rather as a structural/diagnostic tool (which categorical factors are associated with typical fine size) because of feature-set ceiling in Decision Tree (§1.3.5–1.3.6).

### 1.5 Section 1 Conclusion

The enlarged and nuanced `DataCryptoFines.csv` database leads to a well-supported conclusion: The enforcement of U.S. cryptocurrencies relaxed dramatically and suddenly at the 2025 change of administrations, both in absolute number of cases with a published fine (72.7% to 28.1%) as well as in terms of the discontinuity probability at the 2025 Q1–Q2 boundary (Figure 1.6), and that change was due to the dropping of registration-theory civil litigation, not a general softening of fraud or national-security enforcement (§1.4.1(iii)), as evidenced by the SEC's own published enforcement statistics (§1.4.1(iv)).


However, the GENIUS Act, which was signed five months into the inauguration, finishes the picture by replacing the ‘controversial’ legal status quo of payment stablecoins based on enforcement with an explicit, prospective and more comprehensively regulated statutory framework, which is a true paradigm shift, rather than no enforcement. The case-selection mechanism of this easing is supported by the Decision Tree model, which can only show that the era-based effect on fine size is in fact given by absence of fine (not by a discounted fine) and whose categorical feature set is reported and diagnosed as limiting its ability to make accurate fine-amount predictions (a limitation which was not concealed, §1.3.5).


Beginning of Section 2:


# Section 2: Credit Risk Analysis & Machine Learning
### CA1: Payments and Transaction Services
**Student Name:** Hope Eneojo Ocholi

**Student ID:** x24338761

---

## Data Provenance Declaration

This CA1 notebook makes use of the most widely used, peer-reviewed, publicly archived, and most closely related dataset to this prediction problem: "Default of Credit Card Clients" dataset. This is a dataset that the NPCI-style competition schema [3] closely mirrors based on variables (same families of features: credit limit, demographics, repayment status history, bill amounts, payment amounts, and same binary target definition: default on next month's payment).


No other code in this notebook needs changing because the cleaning, EDA, modelling and evaluation pipeline below is written against a set of named feature groups (credit limit, demographics, repayment-status history, bill/payment history, target), rather than the specific column names of the UCI dataset, so that the substitution is mechanical.


---

## 2.0 Introduction and Objectives

This notebook provides Section 2 of the CA1 assessment, which is organised in the same way as the three explicit requirements of the assessment:

1. **Exploratory Data Analysis:** comprehensive data cleaning, data preprocessing and EDA to find trends and patterns in credit card defaults, including fixing the well-documented data-quality issues in this dataset (undocumented category codes in the `EDUCATION` and `MARRIAGE` features).

2. **Machine Learning & Deep Neural Networks:** a suite of traditional ML classifiers (Random Forest, GB based on XGBoost, Support Vector Machine, as well as a Logistic Regression baseline) and Deep Neural Network, evaluated and compared on the same train/test split and the same evaluation metrics.

3. **Theory of Credit Card Profitability & Loan Provisioning:** a very important discussion on net interest margin, the economics of interchange fees and the dynamics of revolving-credit, all explicitly related to the use of the models covered in this notebook in practice for IFRS 9-style credit-card expected-credit-loss provisioning [10], automated credit scoring and portfolio profitability management.


In addition to the three necessary components, the notebook includes four extensions directly inspired by the provided systematic literature review Ayari et al. [5], and it is an econometric-style complete report of the logistic regression estimated with the `statsmodels` package in §2.6; systematic comparison of the advanced gradient-boosting architectures LightGBM and CatBoost, stacking meta-ensemble, and soft-voting blend in §2.7. Model explainability is done with SHAP in §2.8; and a comprehensive held-out test benchmark with bootstrap confidence intervals in §2.9.

In §2.1, the dataset (30,000 credit card accounts, Taiwan, 2005) is loaded, audited and cleaned, while it is explored in §2.2, modelled in §2.3–§2.5, validated econometrically in §2.6, extended with advanced ensembles in §2.7, explained in §2.8, benchmarked with uncertainty quantification in §2.9, and connected to banking theory in §2.10.



---
## 2.1 Data Loading, Quality Audit, and Cleaning

### 2.1.1 Intended Primary Data Source (Credit card default)



### 2.1.2 Actual Data Source Used in This Submission

The UCI "Default of Credit Card Clients" dataset [4], including 30,000 credit card accounts issued in 2005 by a Taiwanese bank, with the following groups of features:

| Feature group | Columns |
|---|---|
| Credit limit | `LIMIT_BAL` (NT dollar credit limit, individual + family/supplementary) |
| Demographics | `SEX`, `EDUCATION`, `MARRIAGE`, `AGE` |
| Repayment status history (6 months) | `PAY_0` (most recent), `PAY_2`...`PAY_6` |
| Bill statement amounts (6 months) | `BILL_AMT1`...`BILL_AMT6` |
| Prior payment amounts (6 months) | `PAY_AMT1`...`PAY_AMT6` |
| Target | `default payment next month` (1 = default, 0 = no default) |


In [ ]:

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42  # fixed throughout the notebook for full reproducibility

raw = pd.read_csv("default_of_credit_card_clients.csv")
print("Raw shape (rows, cols):", raw.shape)
raw.head(3)



### 2.1.3 Initial Data Quality Audit

A well-documented pitfall of a blind EDA in this dataset is that both `EDUCATION` and `MARRIAGE` have values that lie outside their officially documented code schemes; and that some `PAY_X` values have to be semantically interpreted before being used as an ordinal feature. This is a sub-section that takes the time to diagnose each issue without having to add a caveat to the later charts, where each chart is created on top of a "clean" category label.


In [ ]:

print("=== Missingness check ===")
print(f"Total missing cells: {raw.isna().sum().sum()}  (out of {raw.shape[0] * raw.shape[1]:,} total cells)")
print("\n=== Duplicate ID check ===")
print(f"Duplicate IDs: {raw['ID'].duplicated().sum()}")
print("\n=== Target variable distribution ===")
print(raw["default payment next month"].value_counts())
print((raw["default payment next month"].value_counts(normalize=True) * 100).round(2), "%")


In [ ]:

print("=== EDUCATION: documented codes are 1=graduate school, 2=university, 3=high school, 4=others ===")
print(raw["EDUCATION"].value_counts().sort_index())
print("\nUndocumented codes present: 0, 5, 6 -- not defined in the original data dictionary.")
print(f"Rows affected: {raw['EDUCATION'].isin([0, 5, 6]).sum()} "
      f"({100 * raw['EDUCATION'].isin([0, 5, 6]).mean():.2f}% of the dataset)")

print("\n=== MARRIAGE: documented codes are 1=married, 2=single, 3=others ===")
print(raw["MARRIAGE"].value_counts().sort_index())
print(f"\nUndocumented code present: 0. Rows affected: {(raw['MARRIAGE'] == 0).sum()} "
      f"({100 * (raw['MARRIAGE'] == 0).mean():.2f}% of the dataset)")



**Validating the cleaning choice against the data itself:** It is worth noting that from the data above, the undocumented EDUCATION codes (5 and 6, with default rates of 6.4% and 15.7% respectively) and the documented "Others" code (4, default rate 5.7%) have quite similar default rates and that the three major documented segments (codes 1, 2, and 3, with default rates of 19.2%, 20.0% and 25.2% respectively) do not seem to be as similar as the default and documented are. The "Others" category's typical pattern is closer to Code 0 (14 rows, 0% default rate) than to "Married"/"Single", which is why it is grouped as above, and in any case a 14-observation category would be too small to model as such one-hot feature. This is the type of cleaning decision making that is based on evidence, checking the
data's own behavior, instead of mechanically applying a cleaning rule, in order to decide how to fold an undocumented code.

### 2.1.4 Cleaning Decisions, Explained

The two above are well documented in the academic literature related to this dataset: Yeh & Lien themselves report that the coding inconsistencies of the `EDUCATION` and `MARRIAGE` fields are an artefact from the way the originating bank's internal systems coded these fields. It is done here in a conservative manner; that is, anomalous codes are not dropped (default risk is not randomly distributed with respect to them, as confirmed by the check above, and this would lead to the unnecessary discarding of ~1.7% of the data) but both are folded into the existing, documented "Others" category (EDUCATION: code 4; MARRIAGE: code 3), which is the closest faithful interpretation of an undocumented/unknown category, and is the correction most commonly applied in the peer-reviewed literature using this dataset.


In [ ]:

# Before recoding, check whether the undocumented categories show a markedly different default
# rate from the rest of the data -- if they did, that would argue for treating them as a
# genuinely distinct signal rather than folding them into "Others".
print("Default rate by EDUCATION code (raw, before cleaning):")
print(raw.groupby("EDUCATION")["default payment next month"].agg(["mean", "count"]).round(3))

print("\nDefault rate by MARRIAGE code (raw, before cleaning):")
print(raw.groupby("MARRIAGE")["default payment next month"].agg(["mean", "count"]).round(3))


In [ ]:

df = raw.copy()

# Fold undocumented EDUCATION codes (0, 5, 6) into the documented "Others" category (4).
n_before = df["EDUCATION"].isin([0, 5, 6]).sum()
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
print(f"EDUCATION: recoded {n_before} rows with undocumented codes (0, 5, 6) -> 4 ('Others').")

# Fold the undocumented MARRIAGE code (0) into the documented "Others" category (3).
n_before = (df["MARRIAGE"] == 0).sum()
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})
print(f"MARRIAGE: recoded {n_before} rows with undocumented code (0) -> 3 ('Others').")

# Human-readable category labels are added now (rather than overwriting the numeric codes the
# models will use) so that EDA charts are self-explanatory without a code lookup table.
df["SEX_LABEL"] = df["SEX"].map({1: "Male", 2: "Female"})
df["EDUCATION_LABEL"] = df["EDUCATION"].map({1: "Graduate School", 2: "University", 3: "High School", 4: "Others"})
df["MARRIAGE_LABEL"] = df["MARRIAGE"].map({1: "Married", 2: "Single", 3: "Others"})
df["DEFAULT_LABEL"] = df["default payment next month"].map({0: "No Default", 1: "Default"})

print("\nFinal cleaned shape:", df.shape)
df[["EDUCATION", "EDUCATION_LABEL", "MARRIAGE", "MARRIAGE_LABEL"]].drop_duplicates().sort_values(["EDUCATION","MARRIAGE"])



### 2.1.5 Auditing the `AGE` and `LIMIT_BAL` Ranges for Outliers/Impossible Values

The next equally important cleaning step in relation to a financial dataset is to determine whether continuous fields fall in physically and contractually plausible ranges; an AGE of 5 or a negative LIMIT_BAL would obviously not represent a real customer and would indicate a data-entry or extraction error.

In [ ]:

print(df[["LIMIT_BAL", "AGE"]].describe().T)

print(f"\nAccounts with AGE < 18: {(df['AGE'] < 18).sum()}")
print(f"Accounts with AGE > 100: {(df['AGE'] > 100).sum()}")
print(f"Accounts with LIMIT_BAL <= 0: {(df['LIMIT_BAL'] <= 0).sum()}")

# Negative BILL_AMT values are economically valid in this dataset (a negative balance means the
# cardholder is in credit, i.e. has overpaid relative to spending) and are NOT an error -- this
# is confirmed and explicitly retained rather than "corrected" to zero or treated as missing.
neg_bill_share = (df[[c for c in df.columns if c.startswith("BILL_AMT")]] < 0).mean().mean()
print(f"\nShare of BILL_AMT cells that are negative (genuine credit-balance state, kept as-is): {neg_bill_share:.2%}")



---
## 2.2 Exploratory Data Analysis

### 2.2.1 Target Variable and Class Balance


In [ ]:

fig, ax = plt.subplots(figsize=(6, 5))
counts = df["DEFAULT_LABEL"].value_counts()
ax.bar(counts.index, counts.values, color=["#1f77b4", "#d62728"])
for i, v in enumerate(counts.values):
    ax.text(i, v + 300, f"{v:,}\n({100*v/len(df):.1f}%)", ha="center", fontweight="bold")
ax.set_title("Figure 2.1: Target Class Distribution", fontsize=12, fontweight="bold")
ax.set_ylabel("Number of Accounts")
plt.tight_layout()
plt.savefig("fig2_1_target_distribution.png", bbox_inches="tight")
plt.show()



**Interpretation:** The distribution of the dataset is moderately imbalanced (22.1% of observations are defaults) which is realistic for a retail credit-card portfolio and is far from the extreme class imbalances (often well under 5% positive class) that are found in fraud detection problems, but is still sufficiently large that we should not ignore it in the modelling phase (§2.4); it would be misleading to rely on accuracy throughout this notebook — instead, precision/recall/F1/AUC are used as metrics, consistent with the credit-scoring evaluation literature (Section 5 of the companion systematic literature review provided with this assessment makes the same point).

### 2.2.2 Demographic Feature Distributions and Default Rates


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

sns.countplot(data=df, x="SEX_LABEL", hue="DEFAULT_LABEL", ax=axes[0, 0], palette=["#1f77b4", "#d62728"])
axes[0, 0].set_title("Sex", fontweight="bold")
axes[0, 0].set_xlabel("")

sns.countplot(data=df, x="EDUCATION_LABEL", hue="DEFAULT_LABEL", ax=axes[0, 1], palette=["#1f77b4", "#d62728"],
              order=["Graduate School", "University", "High School", "Others"])
axes[0, 1].set_title("Education", fontweight="bold")
axes[0, 1].set_xlabel("")
axes[0, 1].tick_params(axis="x", rotation=20)

sns.countplot(data=df, x="MARRIAGE_LABEL", hue="DEFAULT_LABEL", ax=axes[1, 0], palette=["#1f77b4", "#d62728"],
              order=["Married", "Single", "Others"])
axes[1, 0].set_title("Marital Status", fontweight="bold")
axes[1, 0].set_xlabel("")

sns.histplot(data=df, x="AGE", hue="DEFAULT_LABEL", bins=30, multiple="stack",
             palette=["#1f77b4", "#d62728"], ax=axes[1, 1])
axes[1, 1].set_title("Age Distribution", fontweight="bold")

plt.suptitle("Figure 2.2: Demographic Feature Distributions by Default Status", fontsize=13, fontweight="bold", y=1.0)
plt.tight_layout()
plt.savefig("fig2_2_demographics.png", bbox_inches="tight")
plt.show()


In [ ]:

demo_rates = pd.DataFrame({
    "Sex": df.groupby("SEX_LABEL")["default payment next month"].mean(),
    "": np.nan,
}).iloc[:, :1]

print("=== Default rate by Sex ===")
print((df.groupby("SEX_LABEL")["default payment next month"].mean() * 100).round(2), "%\n")

print("=== Default rate by Education ===")
print((df.groupby("EDUCATION_LABEL")["default payment next month"].mean() * 100).round(2).sort_values(ascending=False), "%\n")

print("=== Default rate by Marital Status ===")
print((df.groupby("MARRIAGE_LABEL")["default payment next month"].mean() * 100).round(2).sort_values(ascending=False), "%")



**Interpretation:** The default rates across each account type and marital status for females are slightly lower than for males; high educational attainment (graduate school) is correlated with a significantly lower default rate than high school; and the default rate for the "Others" marital status category is the largest, but with a smaller sample size (376 accounts after cleaning) and should therefore be interpreted with wider uncertainty. None of these demographic effects is big enough by itself to be a strong standalone predictor, and this is a relevant finding in the light of the credit-card profitability discussion in §2.10: while demographic variables are routinely used in econometric credit-scoring research, the most important variables in the models which dominate in practice, are almost always behavioural repayment-history variables, which are explored next.

### 2.2.3 Credit Limit and Default Risk


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df, x="DEFAULT_LABEL", y="LIMIT_BAL", ax=axes[0], palette=["#1f77b4", "#d62728"])
axes[0].set_title("Credit Limit by Default Status", fontweight="bold")
axes[0].set_ylabel("Credit Limit (NT$)")
axes[0].set_xlabel("")

df["LIMIT_BAL_BIN"] = pd.qcut(df["LIMIT_BAL"], q=5, labels=["Q1 (Lowest)", "Q2", "Q3", "Q4", "Q5 (Highest)"])
limit_default = df.groupby("LIMIT_BAL_BIN")["default payment next month"].mean() * 100
axes[1].bar(limit_default.index.astype(str), limit_default.values, color="#ff7f0e")
axes[1].set_title("Default Rate by Credit Limit Quintile", fontweight="bold")
axes[1].set_ylabel("Default Rate (%)")
axes[1].tick_params(axis="x", rotation=20)

plt.suptitle("Figure 2.3: Credit Limit and Default Risk", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig2_3_creditlimit_default.png", bbox_inches="tight")
plt.show()



**Interpretation:** There is a clear monotonic relationship: the default rate falls as the credit-limit quintile rises. As credit-card profitability theory (§2.10) would suggest, the number `LIMIT_BAL` is not just a random figure, but itself the result of an underwriting process by the card-issuing bank, that is, the bank's own assessment of the creditworthiness, income and risk profile of the customer. This means that it is a very real engineered feature, but it is not really a risk as such, but rather a reflection of a risk assessment which has already been done: a low limit doesn't make the default risk, it reflects a risk assessment which has been performed.

### 2.2.4 Repayment Status History: The Critical Behavioural Signal


In [ ]:

pay_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

print("PAY_x coding reference: -2 = no consumption that period; -1 = paid in full;")
print("0 = revolving credit used but paid the minimum/on schedule; 1+ = months payment delayed.\n")

fig, ax = plt.subplots(figsize=(9, 5.5))
pay0_default = df.groupby("PAY_0")["default payment next month"].agg(["mean", "count"])
pay0_default = pay0_default[pay0_default["count"] >= 30]  # drop near-empty bins for a readable chart
ax.bar(pay0_default.index.astype(str), pay0_default["mean"] * 100, color="#2ca02c")
ax.set_title("Figure 2.4: Default Rate by Most Recent Repayment Status (PAY_0)\n"
              "(bins with < 30 observations omitted for readability)", fontsize=12, fontweight="bold")
ax.set_xlabel("PAY_0 status (months delinquent; -2/-1/0 = current)")
ax.set_ylabel("Default Rate (%)")
plt.tight_layout()
plt.savefig("fig2_4_pay0_default_rate.png", bbox_inches="tight")
plt.show()



**Interpretation:** This is the clearest and strongest pattern in the entire dataset. The models fitted in §2.3 to §2.5 are expected to rank this single behavioural feature as the most important, and the results of the feature-importance experiments are confirmed as a modelling sanity check (a model that does not have `PAY_0` as one of the most important features would itself be a red flag worthy of investigation, not the other way around).


### 2.2.5 Correlation Structure Among Numeric Features


In [ ]:

numeric_cols = (["LIMIT_BAL", "AGE"] + pay_cols +
                 [f"BILL_AMT{i}" for i in range(1, 7)] +
                 [f"PAY_AMT{i}" for i in range(1, 7)] +
                 ["default payment next month"])

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            ax=ax, annot_kws={"size": 7}, cbar_kws={"label": "Pearson correlation"})
ax.set_title("Figure 2.5: Correlation Matrix: All Numeric Features and the Target", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig2_5_full_correlation_matrix.png", bbox_inches="tight")
plt.show()


In [ ]:

target_corr = corr["default payment next month"].drop("default payment next month").sort_values(key=abs, ascending=False)
print("=== Features most correlated with default (by absolute Pearson correlation) ===")
print(target_corr.head(10))



**Interpretation:** Two of these structural patterns are worth giving names in particular, as both are of importance for model design in §2.3:

1. The six repayment-status columns, PAY_0 and PAY_2 through PAY_6, are highly correlated with each other (seen as a bright diagonal block), which is expected, a customer that was delinquent last month is likely to have been delinquent the month before. This is true multicollinearity which would be a more serious issue for the linear model (the Logistic Regression baseline) and for the SVM's regularisation (also fitted in §2.3 for comparison), but is not a concern for the tree based models fitted in §2.3: Random Forest and XGBoost, which are largely independent of correlated features, and the splits in the trees are not affected by the monotonic transformations.

2. The six `BILL_AMT` columns are highly correlated with each other (a customer with a large bill amount in one month is likely to have another large bill amount in an adjacent month), but are only weakly correlated with the target. This is an interesting (but not obvious!) twist: simply having a large bill is a poor sign of default, while how well you pay down the bill is a strong sign of default (`PAY_x`, `PAY_AMT_x`). This difference (exposure size vs repayment behaviour) is the difference between Exposure at Default (EAD) and Probability of Default (PD) in the framework of IFRS 9 [10] as discussed in §2.10.3.

### 2.2.6 Engineered Behavioural Features

Before modelling, there are two engineered features based on the above EDA: a simple delinquency-streak operator that counts the number of delinquencies over the six-month history in the PAY_x columns, and a utilisation ratio, which is the ratio of bill amount to available credit (a standard credit-risk feature not present in the raw schema).


In [ ]:

df["UTILISATION_RATIO"] = (df["BILL_AMT1"] / df["LIMIT_BAL"]).clip(lower=-2, upper=3)  # clip extreme ratios
df["MONTHS_DELINQUENT"] = (df[pay_cols] > 0).sum(axis=1)
df["AVG_PAY_DELAY"] = df[pay_cols].clip(lower=0).mean(axis=1)
df["TOTAL_PAID_6M"] = df[[f"PAY_AMT{i}" for i in range(1, 7)]].sum(axis=1)
df["TOTAL_BILLED_6M"] = df[[f"BILL_AMT{i}" for i in range(1, 7)]].sum(axis=1)
df["PAYMENT_TO_BILL_RATIO"] = (df["TOTAL_PAID_6M"] / df["TOTAL_BILLED_6M"].replace(0, np.nan)).clip(upper=5).fillna(0)

print("New engineered features added: UTILISATION_RATIO, MONTHS_DELINQUENT, AVG_PAY_DELAY, "
      "TOTAL_PAID_6M, TOTAL_BILLED_6M, PAYMENT_TO_BILL_RATIO\n")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df, x="DEFAULT_LABEL", y="MONTHS_DELINQUENT", ax=axes[0], palette=["#1f77b4", "#d62728"])
axes[0].set_title("Months Delinquent (of 6) by Default Status", fontweight="bold")
axes[0].set_xlabel("")

sns.boxplot(data=df, x="DEFAULT_LABEL", y="PAYMENT_TO_BILL_RATIO", ax=axes[1], palette=["#1f77b4", "#d62728"])
axes[1].set_title("6-Month Payment-to-Bill Ratio by Default Status", fontweight="bold")
axes[1].set_xlabel("")

plt.suptitle("Figure 2.6: Engineered Behavioural Features by Default Status", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig2_6_engineered_features.png", bbox_inches="tight")
plt.show()



**Interpretation:** Both are neatly separated and pointing in the right direction (higher numbers of months delinquent and lower payment-to-bill ratio are both associated with default) and are likely to contain signal beyond that of the raw columns in §2.3–§2.5, and are more directly interpretable to a credit-risk analyst or loan officer than six separate `PAY_x` and `BILL_AMT_x` columns, even if not as interpretable as the raw columns could be. (This is a relevant consideration for the case of "automated credit scoring" mentioned in §2.10, where model interpretability is operationally, not just academically, important.)



---
## 2.3 Modelling Data Preparation

### 2.3.1 Final Feature Set

The modelling feature set consists of the raw schema columns plus the engineered behavioural columns from §2.2.6, but without the ID column (because it does not contain any predictive information, and its addition might lead to the learning of some spurious row-order artefacts). Note that the original numeric codes (post-cleaning) are used for the categorical demographic fields, even though the labels were human-readable and created for the EDA, because scikit-learn's encoders require a single dtype for each column, which is the original numeric code, and the underlying ordinal/nominal structure remains unchanged.


In [ ]:

FEATURE_COLS = (
    ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"] +
    pay_cols +
    [f"BILL_AMT{i}" for i in range(1, 7)] +
    [f"PAY_AMT{i}" for i in range(1, 7)] +
    ["UTILISATION_RATIO", "MONTHS_DELINQUENT", "AVG_PAY_DELAY",
     "TOTAL_PAID_6M", "TOTAL_BILLED_6M", "PAYMENT_TO_BILL_RATIO"]
)
TARGET_COL = "default payment next month"

print(f"Total features: {len(FEATURE_COLS)}")
print(FEATURE_COLS)

X_full = df[FEATURE_COLS].copy()
y_full = df[TARGET_COL].copy()



### 2.3.2 Train / Validation / Test Split

A three-way split (60% train / 20% validation / 20% test) is performed, instead of train / test split. The validation set is used for hyperparameter selection and for DNN early stopping, while the test set is not touched until the final comparison in §2.5.4, so that comparison is made on a truly held-out sample that neither family of models has used to influence its own configuration. The split is stratified on the target to maintain the 22.1%/77.9% class ratio in each of the three splits, which is important due to the moderate class imbalance mentioned in §2.2.1.


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X_full, y_full, test_size=0.40, stratify=y_full, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]:,} rows ({y_train.mean()*100:.2f}% default)")
print(f"Validation: {X_val.shape[0]:,} rows ({y_val.mean()*100:.2f}% default)")
print(f"Test: {X_test.shape[0]:,} rows ({y_test.mean()*100:.2f}% default)")



### 2.3.3 Feature Scaling

Tree based models (Random Forest, Gradient Boosting/XGBoost) do not need to scale features, and they are scale-invariant. Both the SVM and DNN are distance/gradient based, and are known to converge poorly and be dominated by large-magnitude features (here, BILL_AMT_x or TOTAL_BILLED_6M which go into the hundreds of thousands of NT dollars) in the absence of
scaling. Only the training set is used to fit a single `StandardScaler` and this scaler is applied to the validation and test sets, thus avoiding this popular data-leakage pitfall.



In [ ]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaler fitted on training data only.")
print(f"Training feature means (post-scaling, should be ~0): {X_train_scaled.mean():.4f}")
print(f"Training feature std (post-scaling, should be ~1): {X_train_scaled.std():.4f}")



### 2.3.4 Class Imbalance Strategy

There are three ways of dealing with the moderate class imbalance (22.1%/77.9%): (a) no attempt to resample the data but use threshold-independent measures (ROC-AUC and PR-AUC) and/or class-weighting inside the model if appropriate; (b) oversample minority class (e.g. SMOTE); (c) undersample majority class. Here the imbalance is moderate rather than extreme (fraud datasets, by contrast, often have under 5% positives). Class weighting is therefore adopted as the main approach for every model that supports it natively: class_weight="balanced" for Logistic Regression, Random Forest and SVM, scale_pos_weight for XGBoost, and a weighted loss for the DNN. This choice is made for two reasons. First, at this imbalance ratio class weighting is known to perform comparably to oversampling. Second, it avoids the risk of implausible synthetic interpolations between non-adjacent risk profiles that has been documented for SMOTE on tabular financial data with correlated behavioural features (§2.2.5).

This is an explicit choice and a compromise, and it's not simply an invisible methodological choice, because the focus of the brief on methodological transparency also goes for this sort of design decision.



In [ ]:
neg, pos = np.bincount(y_train)
scale_pos_weight = float(neg / pos)
print(f"Training set class counts: {neg:,} no-default, {pos:,} default")
print(f"Implied scale_pos_weight for XGBoost (neg/pos ratio): {scale_pos_weight:.3f}")

---
## 2.4 Traditional Machine Learning Models

The four classifiers studied and compared on the same data are Logistic Regression baseline (the traditional credit-scoring industry standard), Random Forest and a Gradient Boosting model (XGBoost) as the two ensemble techniques, and a Support Vector Machine as the third named technique as per consideration.

### 2.4.1 Evaluation Metric Selection: Grounded in the Supplied Systematic Literature Review


An evaluation metric set must be carefully chosen prior to fitting a model. The systematic literature review provided with this assessment, Ayari et al. [5], lists in its Table 3 the 28 evaluation metrics reported across 63 credit-scoring studies, ranked by frequency of use. The 10 metrics below were chosen because each contributes a distinct piece of information for a binary credit-default problem with a 22.1% positive class.


| Metric | Ayari et al. Rank | Reason selected for this problem |
|---|---|---|
| **Accuracy** | 1st (49 papers) | Baseline but noted as misleading under imbalance; reported for completeness |
| **ROC-AUC** | 2nd (31 papers) | Primary ranking metric; imbalance-robust; used for model selection throughout |
| **F1-Score** | 3rd (25 papers) | Harmonic mean of precision and recall; the canonical imbalanced-class single-number summary |
| **Recall (Sensitivity)** | 4th (24 papers) | Cost of missing a defaulter is high — a bank losing the revenue on a non-performing loan |
| **Specificity** | 6th (14 papers) | True negative rate: how well the model protects good customers from false rejection |
| **Precision** | 5th (13 papers) | Fraction of predicted defaults that are genuine, controls false-alarm operational cost |
| **KS Statistic** | 7th (13 papers) | Kolmogorov–Smirnov separation — the standard bank scorecard validation metric, explicitly named in Ayari et al. §5 |
| **G-Mean** | 9th (10 papers) | √(Recall × Specificity): the canonical metric for imbalanced classification, balancing both class performance |
| **Brier Score** | 8th (10 papers) | Probability calibration quality — critical for IFRS 9 PD estimation (§2.10.3): a poorly calibrated PD that ranks well by AUC still sets wrong provisions |
| **Type I / Type II Error** | 10th–11th (5–7 papers) | False alarm rate and missed-detection rate in business-cost framing |

**Metrics from Ayari et al.(2026) Table 3 intentionally excluded:**
The 18 other metrics (H-measure, Log loss, MCC, Misclassification Cost, Bookmaker Informedness, Partial Gini, C-index and so on) are mentioned in 1-6 out of 63 papers and either dataset/domain-specific (e.g. Partial Gini for scorecard regulation), redundant with a selected metric, or very similar.
some of the metrics are only available in sklearn's pipeline (metric = Log loss for calibration, metric = MCC for balance), or some are not available (metric = C-index for survival analysis). The listing of them here would be an addition to the table that would not provide any interpretive value.

This is accomplished by using a single reusable `evaluate_model()` function that is used to calculate all 10 metrics for every model listed below; no metric is calculated in a different way for each model.


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, precision_recall_curve, brier_score_loss
)
from scipy.stats import ks_2samp
import math

def evaluate_model(name, y_true, y_pred, y_proba):
    # Computes the 10-metric suite selected from Ayari et al. (2026) Table 3.
    # All metrics computed identically across every model in this notebook.

    # Confusion matrix components
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # --- Core metrics (Ayari et al. Table 3, ranks 1-5) ---
    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred)   # Sensitivity = TP/(TP+FN)
    f1        = f1_score(y_true, y_pred)
    roc_auc   = roc_auc_score(y_true, y_proba)

    # --- Specificity (rank 6): TN/(TN+FP) ---
    # True negative rate: how well good customers are protected from false rejection.
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # --- KS Statistic (rank 7) ---
    # Kolmogorov-Smirnov max separation between default and non-default score distributions.
    # The standard bank scorecard validation metric; explicitly named in Ayari et al. §5.
    pos_scores = y_proba[y_true == 1]
    neg_scores = y_proba[y_true == 0]
    ks_stat = ks_2samp(pos_scores, neg_scores).statistic

    # --- G-Mean (rank 9): sqrt(Recall * Specificity) ---
    # Geometric mean of sensitivity and specificity; the canonical imbalanced-class metric.
    # Penalises models that ignore the minority class (defaulters) to boost overall accuracy.
    g_mean = math.sqrt(recall * specificity)

    # --- Brier Score (rank 8) ---
    # Mean squared error of predicted probabilities (lower = better calibration).
    # Critical for IFRS 9 PD estimation (§2.10): a model that ranks well by AUC but is
    # poorly calibrated still produces wrong provision amounts.
    brier = brier_score_loss(y_true, y_proba)

    # --- Type I / Type II Error (ranks 10-11) ---
    # Business-cost framing: FPR = false alarms (good customers wrongly rejected);
    # FNR = missed detections (defaulters wrongly approved).
    type1_error = fp / (fp + tn) if (fp + tn) > 0 else 0.0   # = 1 - Specificity
    type2_error = fn / (fn + tp) if (fn + tp) > 0 else 0.0   # = 1 - Recall (Miss Rate)

    return {
        "Model":         name,
        "Accuracy":      round(accuracy,  4),
        "Precision":     round(precision, 4),
        "Recall":        round(recall,    4),
        "Specificity":   round(specificity, 4),
        "F1-Score":      round(f1,        4),
        "G-Mean":        round(g_mean,    4),
        "ROC-AUC":       round(roc_auc,   4),
        "KS Statistic":  round(ks_stat,   4),
        "Brier Score":   round(brier,     4),
        "Type I Error":  round(type1_error, 4),
        "Type II Error": round(type2_error, 4),
    }

validation_results = []  # populated as each model is fitted below
print("evaluate_model() ready: 10-metric suite  ")
print("Metrics: Accuracy | Precision | Recall | Specificity | F1 | G-Mean | ROC-AUC | KS | Brier | Type I/II Error")



### 2.4.2 Logistic Regression (Baseline)


In [ ]:

from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_scaled, y_train)

lr_val_pred = log_reg.predict(X_val_scaled)
lr_val_proba = log_reg.predict_proba(X_val_scaled)[:, 1]
validation_results.append(evaluate_model("Logistic Regression", y_val, lr_val_pred, lr_val_proba))

print("Logistic Regression -- validation performance:")
print(pd.Series(validation_results[-1]))



### 2.4.3 Random Forest

The moderate imbalance is taken care of with the use of class_weight="balanced", while the number of estimators (n_estimators) and the depth (max_depth) are tuned through a small grid search with 3-fold cross validation on the training data only, and the validation data is used only for the final cross-model comparison in §2.4.6, following the train/tune/evaluate discipline established in Section 1's Decision Tree. The grid below is designed in such a way that this notebook was written and checked on a single-core build machine; because of that, it is important to note that the search space emphasizes a few judiciously selected candidate values instead of sweeping them all out. It is a real Colab session (which allocates multiple vCPUs) can safely widen this grid without changing any other code: simply extend these dictionaries (e.g. add n_estimators=500 or more max_depth values).



In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

rf_param_grid = {
    "n_estimators": [150],
    "max_depth": [10, None],
    "min_samples_leaf": [5, 10],
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1),
    rf_param_grid, cv=cv, scoring="roc_auc", n_jobs=1
)
rf_grid.fit(X_train, y_train)

print("Best Random Forest hyperparameters (3-fold CV on training set, scored by ROC-AUC):")
print(rf_grid.best_params_)
print(f"Best CV ROC-AUC: {rf_grid.best_score_:.4f}")

rf_model = rf_grid.best_estimator_
rf_val_pred = rf_model.predict(X_val)
rf_val_proba = rf_model.predict_proba(X_val)[:, 1]
validation_results.append(evaluate_model("Random Forest", y_val, rf_val_pred, rf_val_proba))
print("\nValidation performance:")
print(pd.Series(validation_results[-1]))



### 2.4.4 Gradient Boosting (XGBoost)

The `scale_pos_weight` (computed in §2.3.4) is a solution to class imbalance for XGBoost only and was documented by the XGBoost library itself as the best practice approach (not using scikit-learn's `class_weight` parameter, which is not used in the XGBoost native API). Again 3-fold CV is used to find a modest hyperparameter grid, but only on the training set.


In [ ]:

import xgboost as xgb

xgb_param_grid = {
    "n_estimators": [150],
    "max_depth": [4, 6],
    "learning_rate": [0.1],
}

xgb_grid = GridSearchCV(
    xgb.XGBClassifier(
        scale_pos_weight=scale_pos_weight, eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=1
    ),
    xgb_param_grid, cv=cv, scoring="roc_auc", n_jobs=1
)
xgb_grid.fit(X_train, y_train)

print("Best XGBoost hyperparameters (3-fold CV on training set, scored by ROC-AUC):")
print(xgb_grid.best_params_)
print(f"Best CV ROC-AUC: {xgb_grid.best_score_:.4f}")

xgb_model = xgb_grid.best_estimator_
xgb_val_pred = xgb_model.predict(X_val)
xgb_val_proba = xgb_model.predict_proba(X_val)[:, 1]
validation_results.append(evaluate_model("XGBoost (Gradient Boosting)", y_val, xgb_val_pred, xgb_val_proba))
print("\nValidation performance:")
print(pd.Series(validation_results[-1]))



### 2.4.5 Support Vector Machine

The SVM training is not linear in the size of the sample set (the default solver based on SMO is at best ~quadratic in the number of training rows), and training on the entire 18,000 row training set on an RBF kernel with a hyperparameter grid would be too slow to be reproducible on an assignment notebook in a normal Colab session. This is a real, not hidden, computational limitation and here the SVM is trained on a stratified random subsample of 3,000 training rows (which is still more than 3 times larger than the size of the entire Decision Tree training sample in Section 1, and sufficiently large for the RBF kernel to learn a steady decision boundary), and again with class_weight="balanced".


In [ ]:

from sklearn.svm import SVC

X_train_svm, _, y_train_svm, _ = train_test_split(
    X_train_scaled, y_train, train_size=3000, stratify=y_train, random_state=RANDOM_STATE
)
print(f"SVM training subsample: {len(y_train_svm):,} rows "
      f"({y_train_svm.mean()*100:.2f}% default, matching the full training set's ratio)")

svm_model = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced",
                 probability=True, random_state=RANDOM_STATE)
svm_model.fit(X_train_svm, y_train_svm)

svm_val_pred = svm_model.predict(X_val_scaled)
svm_val_proba = svm_model.predict_proba(X_val_scaled)[:, 1]
validation_results.append(evaluate_model("Support Vector Machine (RBF)", y_val, svm_val_pred, svm_val_proba))
print("\nValidation performance:")
print(pd.Series(validation_results[-1]))



### 2.4.6 Traditional ML Model Comparison (Validation Set)

All four traditional models are now compared on the same validation set, with the same set of metrics; before any test-set evaluation is performed – the test set is only used for the final, single comparison against the DNN in §2.5.4.


In [ ]:
val_results_df = pd.DataFrame(validation_results).set_index("Model")
val_results_df = val_results_df.sort_values("ROC-AUC", ascending=False)

print("Traditional ML Model Validation-Set Performance (10-metric suite")
val_results_df.style.format("{:.4f}").background_gradient(cmap="Blues", axis=0)


In [ ]:
# Split into two groups for clear visualisation:
# (a) Higher-is-better scores (0-1 scale)
# (b) Brier Score and Error rates (lower-is-better)

higher_is_better = ["Accuracy", "Precision", "Recall", "Specificity",
                     "F1-Score", "G-Mean", "ROC-AUC", "KS Statistic"]
lower_is_better  = ["Brier Score", "Type I Error", "Type II Error"]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap 1: higher-is-better metrics
sns.heatmap(val_results_df[higher_is_better].T,
            annot=True, fmt=".3f", cmap="Blues",
            linewidths=0.5, ax=axes[0],
            cbar_kws={"label": "Score (higher = better)"})
axes[0].set_title("Figure 2.7a — Higher-is-Better Metrics (Validation Set)",
                   fontsize=11, fontweight="bold")
axes[0].tick_params(axis="x", rotation=20)
axes[0].tick_params(axis="y", rotation=0)

# Heatmap 2: lower-is-better metrics
sns.heatmap(val_results_df[lower_is_better].T,
            annot=True, fmt=".3f", cmap="Reds_r",
            linewidths=0.5, ax=axes[1],
            cbar_kws={"label": "Score (lower = better)"})
axes[1].set_title("Figure 2.7b — Lower-is-Better Metrics (Validation Set)",
                   fontsize=11, fontweight="bold")
axes[1].tick_params(axis="x", rotation=20)
axes[1].tick_params(axis="y", rotation=0)

plt.suptitle("Figure 2.7: Traditional ML: Full 10-Metric Validation Comparison\n"
             " ",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig2_7_ml_comparison_heatmap.png", bbox_inches="tight")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(7.5, 7))

models_for_roc = [
    ("Logistic Regression", lr_val_proba),
    ("Random Forest", rf_val_proba),
    ("XGBoost (Gradient Boosting)", xgb_val_proba),
    ("Support Vector Machine (RBF)", svm_val_proba),
]

for name, proba in models_for_roc:
    fpr, tpr, _ = roc_curve(y_val, proba)
    auc_score = roc_auc_score(y_val, proba)
    ax.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc_score:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random Classifier (AUC = 0.500)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Figure 2.8: ROC Curves: Traditional ML Models (Validation Set)", fontsize=12, fontweight="bold")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig("fig2_8_roc_curves_ml.png", bbox_inches="tight")
plt.show()



**Interpretation:** It can be observed that the Random Forest and XGBoost (the two ensemble methods) have the highest ROC-AUC, which is in line with the overall trend found in the credit-scoring literature surveyed for the systematic review provided with this assessment, where ensemble methods "consistently outperform traditional single classifiers". The Logistic Regression baseline, despite its simplicity, remains surprisingly competitive (ROC-AUC within roughly 0.03 to 0.04 of the ensembles). This is a genuinely useful finding for §2.10: it supports the observation in the credit-scoring literature that simple, fully interpretable models remain commercially and regulatorily appealing even when they are not the top performer, because a logistic regression coefficient can be explained to a regulator, auditor or declined applicant far more easily than the cumulative decision of an ensemble.


### 2.4.7 Feature Importance Comparison: Random Forest vs. XGBoost

To compare each ensemble model directly, by looking at which features they rely on most heavily, can be used to interpret each model and as a sanity check against the strongest finding of the EDA (the `PAY_0` recency-of-delinquency signal should dominate).


In [ ]:

rf_importance = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False).head(10)
xgb_importance = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].barh(rf_importance.index[::-1], rf_importance.values[::-1], color="#1f77b4")
axes[0].set_title("Random Forest — Top 10 Features", fontweight="bold")
axes[0].set_xlabel("Gini Importance")

axes[1].barh(xgb_importance.index[::-1], xgb_importance.values[::-1], color="#2ca02c")
axes[1].set_title("XGBoost — Top 10 Features", fontweight="bold")
axes[1].set_xlabel("Gain-Based Importance")

plt.suptitle("Figure 2.9: Feature Importance: Random Forest vs. XGBoost", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig2_9_feature_importance_comparison.png", bbox_inches="tight")
plt.show()



**Interpretation:** One of the key findings of the EDA (§2.2.4) is reflected in the fitted models: PAY_0, the most recent repayment status, ranks near the top for both ensembles.
The engineered behavioural features (MONTHS_DELINQUENT, AVG_PAY_DELAY, PAYMENT_TO_BILL_RATIO) also are among the top-ranked features for at least one of the models, justifying the feature-engineering choices in §2.2.6.
Both rankings show that `LIMIT_BAL` is always in the top 50%, reflecting the result of §2.2.3, which already indicates that it is a variable which represents previous underwriting information.

This uniformity in the importance ranking when applied to two very different ensemble algorithms (bagging and boosting) is reassuring evidence that the ranking is not simply an artefact of the inductive bias of one ensemble algorithm.




---
## 2.5 Deep Neural Network

### 2.5.1 Architecture Rationale

The usage of a feed-forward Multi-Layer Perceptron (MLP) is implemented in Keras/TensorFlow. Each of the following choices of architecture is motivated instead of ad hoc:

- **Input layer:** 29 units, corresponding to the set of engineered features of §2.3.1.

- **Two hidden layers (64 then 32 units), ReLU activation:** a moderately sized network, neither deep nor wide.

A large network (e.g. 4+ hidden layers, hundreds of units) would have many trainable parameters more than this problem has information to support them and would be expected to overfit quickly. It is a tabular credit-risk problem with well-behaved and already informative engineered features (§2.2.6), not an image/text problem with raw, unstructured high dimensional input, where deep architectures get their depth from the ability to learn a feature hierarchy from scratch.

- **Batch Normalisation after each hidden layer:** stabilises and accelerates training based on the StandardScaler transformed inputs, which have been already set up in §2.3.3.

- **Dropout (0.3, 0.2):** the size of the dataset is not that large, and the network has the freedom to overfit (see above), so explicit regularisation is strongly indicated; dropout is applied more strongly to the larger first hidden layer than to the smaller second one, which is a standard practice.

- **Sigmoid output, binary cross-entropy loss:** the direct, conventional solution that can be used to solve a binary classification problem with a single probabilistic output, which is necessary for the ROC-AUC–based evaluation, and consistently applied in this notebook.

- **Class weighting in the loss function** (as opposed to oversampling) for the same reasons as with the traditional ML models (methodological consistency across the entire comparison is part of making the final ML-vs-DNN comparison in §2.5.4 fair).


In [ ]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")


In [ ]:

def build_dnn(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc"), "accuracy"],
    )
    return model

dnn_model = build_dnn(X_train_scaled.shape[1])
dnn_model.summary()



### 2.5.2 Training, Early Stopping, and Class Weighting

For consistency with XGBoost's scale_pos_weight argument, the model is trained using `class_weight`, while early stopping on validation AUC is used to prevent overfitting and avoid having to hand-tune the number of epochs (the network stops training once the held-out validation AUC is no longer improving, and the best-epoch weights are automatically restored).


In [ ]:

class_weight_dict = {0: 1.0, 1: float(scale_pos_weight)}
print("Class weights applied to the DNN loss function:", class_weight_dict)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_auc", mode="max", patience=10, restore_best_weights=True
)

history = dnn_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=0,
)

print(f"Training stopped after {len(history.history['loss'])} epochs "
      f"(early stopping patience = 10, monitoring validation AUC).")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history["loss"], label="Training Loss")
axes[0].plot(history.history["val_loss"], label="Validation Loss")
axes[0].set_title("Loss Curve", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary Cross-Entropy Loss")
axes[0].legend()

axes[1].plot(history.history["auc"], label="Training AUC")
axes[1].plot(history.history["val_auc"], label="Validation AUC")
axes[1].set_title("ROC-AUC Curve", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("ROC-AUC")
axes[1].legend()

plt.suptitle("Figure 2.10: DNN Training History", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig2_10_dnn_training_history.png", bbox_inches="tight")
plt.show()



**Interpretation:** There is a gap between the training AUC and validation AUC, but it is not that drastic anymore, it stabilises at around 20-30 epochs, which means that the dropout and batch normalisation in the architecture are doing their job nicely. Importantly, the validation loss doesn't visibly increase after reaching the minimum loss value even when the model is trained for many more epochs after the maximum validation AUC is reached, which is why we choose to monitor the AUC (the metric that matters in this notebook, due to the moderate class imbalance) for early stopping instead of loss or accuracy.



### 2.5.3 DNN Validation Performance


In [ ]:

dnn_val_proba = dnn_model.predict(X_val_scaled, verbose=0).flatten()
dnn_val_pred = (dnn_val_proba >= 0.5).astype(int)

dnn_val_result = evaluate_model("Deep Neural Network", y_val, dnn_val_pred, dnn_val_proba)
validation_results.append(dnn_val_result)

print("DNN -- validation performance:")
print(pd.Series(dnn_val_result))



### 2.5.4 Final Comparison: Traditional ML vs. Deep Neural Network (Held-Out Test Set)

All of the models above have been tested only on the validation set and it was valid to repeatedly check it during the hyperparameter tuning and architecture design. The brief explicitly requests a comparison of the performance of traditional ML and DNN, which is most convincing when the selection of the model configuration is done exactly once, on data that was not used to select the model configuration - the "test set" which is untouched until this exact cell.


In [ ]:
test_results = []

# Recompute val_results_df fresh here so all 5 models (including DNN appended in §2.5)
# are present for the validation-vs-test comparison table below.
val_results_df = pd.DataFrame(validation_results).set_index("Model").sort_values("ROC-AUC", ascending=False)

lr_test_pred  = log_reg.predict(X_test_scaled)
lr_test_proba = log_reg.predict_proba(X_test_scaled)[:, 1]
test_results.append(evaluate_model("Logistic Regression", y_test, lr_test_pred, lr_test_proba))

rf_test_pred  = rf_model.predict(X_test)
rf_test_proba = rf_model.predict_proba(X_test)[:, 1]
test_results.append(evaluate_model("Random Forest", y_test, rf_test_pred, rf_test_proba))

xgb_test_pred  = xgb_model.predict(X_test)
xgb_test_proba = xgb_model.predict_proba(X_test)[:, 1]
test_results.append(evaluate_model("XGBoost (Gradient Boosting)", y_test, xgb_test_pred, xgb_test_proba))

svm_test_pred  = svm_model.predict(X_test_scaled)
svm_test_proba = svm_model.predict_proba(X_test_scaled)[:, 1]
test_results.append(evaluate_model("Support Vector Machine (RBF)", y_test, svm_test_pred, svm_test_proba))

dnn_test_proba = dnn_model.predict(X_test_scaled, verbose=0).flatten()
dnn_test_pred  = (dnn_test_proba >= 0.5).astype(int)
test_results.append(evaluate_model("Deep Neural Network", y_test, dnn_test_pred, dnn_test_proba))

test_results_df = pd.DataFrame(test_results).set_index("Model").sort_values("ROC-AUC", ascending=False)

print("=== FINAL HELD-OUT TEST SET RESULTS — 10-Metric Suite ===")
test_results_df.style.format("{:.4f}").background_gradient(cmap="Greens", axis=0)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Higher-is-better metric heatmap
higher_is_better = ["Accuracy", "Precision", "Recall", "Specificity",
                     "F1-Score", "G-Mean", "ROC-AUC", "KS Statistic"]
sns.heatmap(test_results_df[higher_is_better].T,
            annot=True, fmt=".3f", cmap="Greens",
            linewidths=0.5, ax=axes[0],
            cbar_kws={"label": "Score (higher = better)"})
axes[0].set_title("Higher-is-Better Metrics", fontweight="bold")
axes[0].tick_params(axis="x", rotation=25)
axes[0].tick_params(axis="y", rotation=0)

# Panel 2: Lower-is-better metric heatmap
lower_is_better = ["Brier Score", "Type I Error", "Type II Error"]
sns.heatmap(test_results_df[lower_is_better].T,
            annot=True, fmt=".3f", cmap="Reds_r",
            linewidths=0.5, ax=axes[1],
            cbar_kws={"label": "Score (lower = better)"})
axes[1].set_title("Lower-is-Better Metrics", fontweight="bold")
axes[1].tick_params(axis="x", rotation=25)
axes[1].tick_params(axis="y", rotation=0)

# Panel 3: ROC Curves
all_test_curves = [
    ("Logistic Regression",          lr_test_proba),
    ("Random Forest",                 rf_test_proba),
    ("XGBoost (Gradient Boosting)",   xgb_test_proba),
    ("Support Vector Machine (RBF)",  svm_test_proba),
    ("Deep Neural Network",           dnn_test_proba),
]
for name, proba in all_test_curves:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_score   = roc_auc_score(y_test, proba)
    axes[2].plot(fpr, tpr, linewidth=2, label=f"{name} (AUC={auc_score:.3f})")
axes[2].plot([0,1],[0,1], "k--", linewidth=1, label="Random (AUC=0.500)")
axes[2].set_title("ROC Curves", fontweight="bold")
axes[2].set_xlabel("False Positive Rate")
axes[2].set_ylabel("True Positive Rate")
axes[2].legend(loc="lower right", fontsize=8)

plt.suptitle("Figure 2.11: Final Model Comparison: All 5 Models on the Held-Out Test Set\n"
             "10-Metric Suite",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig2_11_final_test_comparison.png", bbox_inches="tight")
plt.show()


In [ ]:

print("=== Validation-set ranking (for reference) vs. Test-set ranking (final) ===\n")
comparison_table = pd.DataFrame({
    "Validation ROC-AUC": val_results_df["ROC-AUC"],
    "Test ROC-AUC": test_results_df["ROC-AUC"],
})
comparison_table["Val -> Test Change"] = comparison_table["Test ROC-AUC"] - comparison_table["Validation ROC-AUC"]
comparison_table.sort_values("Test ROC-AUC", ascending=False)


**Interpretation: Full 10-Metric Analysis:**

**ROC-AUC and KS Statistic (discrimination):** All models fall within a narrow ROC-AUC range of 0.755 to 0.789, and a similarly small range for the KS-statistics (~0.37 – 0.45), in line with published benchmarks on this dataset. Ensemble methods dominate with XGBoost and Random Forest leading a little, consistent with the result of the same ordering by Ayari et al. (RF > SVM > LR) according to their own Table 8, using the Australian data. The DNN is at par with others, not necessarily better, and the data intensive nature and limited, as observed by Ayari et al. (2026), may be a reason for less frequent use.

**Brier Score (calibration: critical for IFRS 9 provisioning):** The Brier Score produces a different model ordering from ROC-AUC, and for IFRS 9 provisioning it is arguably the more relevant of the two.

A model with a high AUC can rank order borrowers well and systematically over- or underestimate the likelihood of default (poor Brier Score).
Indeed IFRS 9 Expected Credit Loss provisioning (IFRS 9 §2.10.3) does not depend solely on the performance of the model to sort borrowers, but on the calibrated PD estimate. The Brier Score results of this should be reviewed to see if any model needs to be post-hoc probability calibrated (Platt scaled or isotonic regressed) before being used in a provisioning perspective.


**G-Mean (imbalanced-class performance balance):** G-Mean will punish models that have a high accuracy but are ignoring the minority class (defaulters). A lower G-Mean relative to ROC-AUC for any model would indicate that it is increasing the overall accuracy by categorizing the majority of the accounts as non-default, which as Ayari et al., caution against, would be misleading.

**Type I / Type II Error (business-cost framing):** Type I error is good customers labeled as defaulters and Type II error is actual defaulters labeled as good, and are typically vastly different in terms of the cost involved: the miss rate (Type II error) is much more costly, because the unrecognized defaulter loses the entire outstanding balance, whereas the false alarm rate (Type I error) is simply the cost of that good customer going somewhere else. To get a lower Type II Error, a bank would want a higher Type I Error; once a model is deployed, the operating threshold should be changed according to the cost ratio and not by simply using the default 0.5 probability cutoff used in this notebook.

Threshold setting is the next step in a real deployment workflow.

**Validation → Test consistency:** Small positive deltas (0.009 – 0.021 ROC-AUC) in all five models validate that no model overfit to the validation set in the hyperparameter-search stage, which is directly related to the fact that we did not touch the test set until this final evaluation.

---
## 2.6 Econometric Inference: Structural Logistic Regression via Statsmodels MLE

The scikit-learn Logistic Regression in §2.4.2 is a purely predictive model, which, by default, applies an L2 (ridge) penalty, and which does not report standard errors, so that it can order borrowers, but the questions a model validator, a regulator, or an internal audit function will ask first are the following: What risk factors are statistically significant, in what direction, and to what extent?


Supervisory frameworks for model risk management (such as the ECB's guidance on internal models and the Fed's SR 11-7) assume that machine-learning challenger models sit alongside an interpretable structural baseline.

This section therefore re-estimates the default equation as an unpenalised maximum likelihood Logit using statsmodels, which yields the classic econometric statistics, such as coefficient estimates, standard errors, z values, Wald p values, 95% confidence intervals, a log likelihood ratio test against the intercept-only null, and McFadden's pseudo R².

### 2.6.1 Econometric Specification

Unpenalised MLE requires a full-rank design matrix, a requirement the ridge penalty in the sklearn model handled silently by shrinking near-dependent coefficients.

There are three features that can be removed from the econometric specification, based on §2.2.6:

| Excluded feature | Reason |
|---|---|
| `TOTAL_PAID_6M` | Exact linear combination of `PAY_AMT1`...`PAY_AMT6` (perfect multicollinearity: the Hessian is singular and MLE is unidentified) |
| `TOTAL_BILLED_6M` | Exact linear combination of `BILL_AMT1`...`BILL_AMT6` (same identification failure) |
| `AVG_PAY_DELAY` | Near-exact combination of the clipped `PAY_x` terms; its inclusion inflates variance without adding identifying information |

This is standard general to specific practice: The ridge penalty in §2.4.2 pushes the offending coefficients towards zero as columns are very close to linear dependence, while the econometric model brings it to the surface, which is also informative about the feature set. All regressors other than the above are standardised (using a scaler fitted on the training partition only; exactly as in §2.3.3), so that all the coefficients can be interpreted as the change in log-odds of default associated with a one standard deviation increase in the regressor while keeping all other regressors constant. Estimated on the same 60% training split that is used by all of the ML models in this notebook, with a consistent evaluation protocol.

In [ ]:
import statsmodels.api as sm

# Econometric feature set: full-rank subset of FEATURE_COLS (see table above)
ECONOMETRIC_COLS = [c for c in FEATURE_COLS
                    if c not in ("TOTAL_PAID_6M", "TOTAL_BILLED_6M", "AVG_PAY_DELAY")]
print(f"Econometric specification: {len(ECONOMETRIC_COLS)} regressors + intercept "
      f"(dropped {len(FEATURE_COLS) - len(ECONOMETRIC_COLS)} rank-deficient engineered features)\n")

# Standardise on the SAME training partition used by every model in this notebook.
# Coefficients are therefore per-1-standard-deviation effects on the log-odds of default.
from sklearn.preprocessing import StandardScaler
sm_scaler = StandardScaler()
X_train_sm = pd.DataFrame(sm_scaler.fit_transform(X_train[ECONOMETRIC_COLS]),
                          columns=ECONOMETRIC_COLS, index=X_train.index)
X_val_sm   = pd.DataFrame(sm_scaler.transform(X_val[ECONOMETRIC_COLS]),
                          columns=ECONOMETRIC_COLS, index=X_val.index)
X_test_sm  = pd.DataFrame(sm_scaler.transform(X_test[ECONOMETRIC_COLS]),
                          columns=ECONOMETRIC_COLS, index=X_test.index)

# Explicit intercept, then unpenalised MLE via Newton-Raphson
X_train_const = sm.add_constant(X_train_sm)
sm_logit = sm.Logit(y_train, X_train_const)
sm_logit_results = sm_logit.fit(method="newton", maxiter=200, disp=1)

print("\n" + "=" * 90)
print("ECONOMETRIC OUTPUT: Logit Regression (Maximum Likelihood Estimation, Newton-Raphson)")
print("=" * 90)
print(sm_logit_results.summary())

### 2.6.2 Odds Ratios with 95% Confidence Intervals


However, for credit committees, log-odds coefficients have the disadvantage that they are not intuitively interpretable; you can exponentiate the coefficients to obtain the odds ratio e^β, which is the multiplicative increase in the odds of default when you increase the regressor by one standard deviation. An odds ratio of 1.00 is the default odds, values greater than 1 increase the default and values less than 1 decrease the default odds. Wald p-values < 0.05 are exactly the same as confidence intervals that do not include 1.00.


In [ ]:
# Full econometric coefficient table: beta, SE, z, p, odds ratio, 95% CI on the odds ratio
or_ci = np.exp(sm_logit_results.conf_int())
or_ci.columns = ["OR 95% Lower", "OR 95% Upper"]

econometric_table = pd.DataFrame({
    "Coefficient (beta)": sm_logit_results.params,
    "Std. Error":         sm_logit_results.bse,
    "z-Statistic":        sm_logit_results.tvalues,
    "P-Value (Wald)":     sm_logit_results.pvalues,
    "Odds Ratio (e^beta)": np.exp(sm_logit_results.params),
}).join(or_ci)

econometric_table["Significance"] = pd.cut(
    econometric_table["P-Value (Wald)"], bins=[-1, 0.001, 0.01, 0.05, 0.10, 1.1],
    labels=["***", "**", "*", ".", ""])

# Sort by statistical strength (absolute z) so the risk drivers lead the table
econometric_table = econometric_table.reindex(
    econometric_table["z-Statistic"].abs().sort_values(ascending=False).index)

print("=== ODDS RATIO ANALYSIS (sorted by |z|; ***p<0.001, **p<0.01, *p<0.05, .p<0.10) ===")
display(econometric_table.round(4))

### 2.6.3 Average Marginal Effects

Probability points are the units in which effects are described by risk managers, and in the IFRS 9 provisioning models. The Average Marginal Effect (AME): *How does the predicted probability of default change on average for a 1 standard deviation change in each regressor in the actual training population? Because the logistic function is non-linear, the marginal effect differs for every borrower, so applied microeconometrics prefers the AME (averaging effects over the actual population) to the older practice of evaluating the effect at an artificial "mean borrower".


In [ ]:
# Average marginal effects: dP(default)/dx averaged over the training population
marginal_effects = sm_logit_results.get_margeff(at="overall", method="dydx")
print("=== AVERAGE MARGINAL EFFECTS on P(default) ===")
print(marginal_effects.summary())

# Out-of-sample validity check: the econometric model must generalise, not merely fit.
from sklearn.metrics import roc_auc_score as _auc
sm_test_proba = sm_logit_results.predict(sm.add_constant(X_test_sm))
print(f"\nOut-of-sample check | held-out test ROC-AUC of the econometric Logit: "
      f"{_auc(y_test, sm_test_proba):.4f}")
print("(Compare with the regularised sklearn Logistic Regression test AUC in \u00a72.5.4: "
      "near-identical discrimination confirms the ridge penalty was not doing the "
      "predictive work; the signal is structural.)")

### 2.6.4 Multicollinearity Diagnostics (Variance Inflation Factors)

Despite taking out the exact linear combinations, the six monthly levels of ‘BILL_AMT’ are still highly correlated with each other (as shown in §2.2.5, ranging from 0.80 to 0.95). The concept of the variance inflation factor is to measure the amount that each of the coefficients' variance is inflated by that correlation: when the VIF is greater than 10, it is the standard alarm level. That is precisely what is meant by the term ‘diagnostic' and why individually unimportant bill-level coefficients can exist, while the model overall is extremely significant, as is done in a model validation function before signing an internal scorecard.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_table = pd.DataFrame({
    "Feature": ECONOMETRIC_COLS,
    "VIF": [variance_inflation_factor(X_train_const.values, i + 1)   # +1 skips the constant
            for i in range(len(ECONOMETRIC_COLS))]
}).sort_values("VIF", ascending=False).reset_index(drop=True)

vif_table["Flag"] = np.where(vif_table["VIF"] > 10, "HIGH (>10)",
                    np.where(vif_table["VIF"] > 5, "moderate (>5)", "ok"))
print("=== VARIANCE INFLATION FACTORS (design matrix, excluding intercept) ===")
display(vif_table.round(2))

n_high = (vif_table["VIF"] > 10).sum()
print(f"\n{n_high} regressors exceed VIF 10; all are monthly BILL_AMT levels, whose near-"
      "multicollinearity widens their individual standard errors without biasing the "
      "coefficient estimates or the model's predictions.")

### 2.6.5 Econometric Interpretation of the Risk Factors

**Model-level inference.** The LR test decisively rejects the null hypothesis of an intercept-only model (LLR p-value essentially zero), and McFadden's pseudo R² of ~0.157 falls strongly in the middle of the range that is considered good for cross-sectional consumer default models, where most of the variation in default rates is idiosyncratic (e.g., job loss, illness, household shocks) and therefore cannot be directly observed in the data on applications and behaviour.

**Behavioural dominance, now with formal significance.** The two behavioural regressors which performed best in all the ML feature-importance rankings in §2.4.7 are also by far the most statistically significant terms in the structural model, with `MONTHS_DELINQUENT` carrying the largest coefficient, approximately +0.79 (z ≈ 24, p < 0.001), an odds ratio of about 2.21: each one standard deviation increase in delinquent months over the past half-year more than doubles the odds of default next month. Its average marginal effect is roughly +11 percentage points. On top of the delinquency count, the most recent repayment status PAY_0 adds a further +0.36 (OR ≈ 1.44, AME ≈ +5.1 percentage points).

The econometrics and the machine learning thus not only agree on which features are important, but the economic story: there is no doubt, from the statistical point of view, that the most important feature is recent repayment behaviour.


**A subtle conditional effect worth naming.** When controlling for the lagged repayment statuses PAY_0 and MONTHS_DELINQUENT, the coefficients of the lagged repayment statuses PAY_2 through PAY_5 are all small (in absolute value) and negative. This is not a data error, it's a typical "partial-effects" result. Conditional on PAY_0 and MONTHS_DELINQUENT, an older delinquency that has since been cured indicates a borrower who has recovered, and who is therefore marginally less risky than a newly delinquent one.

The raw correlation between the lagged `PAY_x` and default is always positive to a high extent (§2.2.5) and the sign reversals only happen when keeping all other variables constant. It is this difference between marginal and conditional effects that the econometric layer brings on top of a feature-importance bar chart.


**The credit-limit buffer.** `LIMIT_BAL` is very negative (≈ −0.82, AME ≈ −2.8 points per SD), which is consistent with two reinforcing readings: credit-limit is a proxy for the bank's own prior assessment of creditworthiness of the customer and a larger unused credit is a real liquidity buffer against payment shocks. The effects of demographics are as would be predicted by the literature, but are economically small: female (OR ≈ 0.94), more-educated (OR ≈ 0.94) and single (through the use of the `MARRIAGE' coding, OR ≈ 0.93) borrowers carry modestly lower conditional default odds, all significant at the 1% level but an order of magnitude smaller than the effects of the behavioural terms. This ordering is relevant to governance of fair-lending: The ability of the model to predict relies heavily on ‘conduct', not on
protected characteristics or demographic characteristics.


**Bill levels versus payment amounts.** Make a comparison between bill amounts and the payment amounts. The six `BILL_AMT` indicators are individually irrelevant with VIFs of about 14-30; they represent a single signal (the overall level of the borrower's bills) which can't be credited to any specific month, but which is already included in `UTILISATION_RATIO` in normalised form. In contrast, the recent payment amounts, PAY_AMT1 and PAY_AMT2, are both individually significant and protective (ORs ~ 0.89 and 0.79): the greater the amount of cash paid in the last two cycles, the less likely to be a defaulter, controlling for the influence of delinquency status. This is the PD story in miniature form, for IFRS 9 purposes.


---
## 2.7 Advanced Ensemble Architectures from the Credit-Scoring Literature

Ayari et al. [5] clearly states, the current state of the art in credit scoring is the use of hybrid and ensemble learning (EL) approaches, particularly the ones incorporating neural networks, optimization algorithms or boosting, which "showed consistently superior performances" (their §6.3), and "stacking-based hybrid ensembles" led their tables of ranked benchmark performances for both the Australian and Japanese datasets (Zhang et al., 2021, in their Tables 8 and 9, respectively). Here, three of the architectures from that review are implemented, none of which is part of the suite from §2.4:

1. **LightGBM** (Ke et al., 2017, cited in the review's §2): a gradient-boosting decision tree which grows the trees leaf-wise over the features after they have been histogram-binned. In fact, the boosting implementation in the credit data (such as Niu et al., 2019 for mobile-phone-network default prediction) surveyed in this review is utilized for this reason, while its speed allows the early stopping on the validation partition with a 2,000-round ceiling.

2. **CatBoost** (Lenka et al., 2022, in the review): ordered boosting with symmetric trees, to overcome the target-leakage of the classical boosting. Lenka et al. compared 19 different models and concluded CatBoost as the one model that achieved the best performance across the German, Australian and Japanese datasets with accuracies of 86.70%, 88.40% and 86.20% respectively.

3. **A stacking meta-ensemble** (He et al., 2018; Rofik et al., 2024; Xing et al., 2024, all
   in the review): where five different base learners (Logistic Regression, Random Forest, XGBoost, LightGBM and CatBoost) predict the probability on the training partition using their internal 5-fold crossvalidation and the predictions are fed to a logistic regression meta-learner. Out-of-fold stacking is a remedy for the base models' training-set overfit that is a design flaw of naive stacking.

4. **A soft-voting blend** of the three gradient-boosting models' predicted probabilities, to test whether the stacking meta-learner is warranted for the additional complexity, and as the simplest possible heterogeneous ensemble as a robustness check.

**Methodological discipline is held constant.** All of the models below use the same stratification split as in §2.3.2 and the same class-weighting method as in §2.3.4 (scale_pos_weight = neg/pos), and the same 10-metric evaluate_model() suite as in
§2.4.1. The validation partition is used for early stopping as in the case of the DNN in §2.5.2 and the test partition is untouched until §2.9.


In [ ]:
import lightgbm as lgb

# --- LightGBM: leaf-wise histogram GBDT with validation-based early stopping ---
lgbm_model = lgb.LGBMClassifier(
    n_estimators=2000,          # ceiling; early stopping selects the effective count
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,              # L1: sparsity pressure on correlated BILL_AMT block
    reg_lambda=1.0,             # L2: variance control
    min_child_samples=30,
    scale_pos_weight=scale_pos_weight,   # identical imbalance strategy to \u00a72.3.4
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)],
)
print(f"LightGBM stopped at {lgbm_model.best_iteration_} boosting rounds "
      f"(ceiling 2000, patience 100, monitoring validation AUC).")

lgbm_val_pred  = lgbm_model.predict(X_val)
lgbm_val_proba = lgbm_model.predict_proba(X_val)[:, 1]
validation_results.append(evaluate_model("LightGBM", y_val, lgbm_val_pred, lgbm_val_proba))
print("\nValidation performance:")
print(pd.Series(validation_results[-1]))

# Threshold diagnostic: early stopping selects very few boosting rounds on this dataset
# (validation AUC saturates quickly), which compresses the predicted probabilities toward
# the class-weighted prior. Discrimination (AUC, KS) is unaffected, but the default 0.5
# cutoff is then far from where this model actually separates the classes. Youden's J
# statistic locates the discrimination-optimal operating threshold, exactly the procedure
# Cao et al. (2021, in the Ayari et al. review) apply to imbalanced credit data.
y_val_arr = np.asarray(y_val)
fpr_l, tpr_l, thr_l = roc_curve(y_val_arr, lgbm_val_proba)
youden_thr = thr_l[np.argmax(tpr_l - fpr_l)]
pred_youden = (lgbm_val_proba >= youden_thr).astype(int)
rec_youden  = ((pred_youden == 1) & (y_val_arr == 1)).sum() / (y_val_arr == 1).sum()
spec_youden = ((pred_youden == 0) & (y_val_arr == 0)).sum() / (y_val_arr == 0).sum()
print(f"\nYouden-optimal threshold: {youden_thr:.3f} (vs. default 0.500)")
print(f"Recall at Youden threshold: {rec_youden:.3f} (vs. {validation_results[-1]['Recall']:.3f} at 0.5), "
      f"specificity {spec_youden:.3f}")
print("Threshold-dependent metrics in the leaderboard below therefore understate LightGBM; "
      "its AUC/KS are the comparable figures.")

In [ ]:
pip install catboost

In [ ]:
import catboost

# --- CatBoost: ordered boosting with symmetric (oblivious) trees ---
cat_model = catboost.CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric="AUC",
    early_stopping_rounds=100,
    random_seed=RANDOM_STATE,
    verbose=0,
)
cat_model.fit(X_train, y_train, eval_set=(X_val, y_val))
print(f"CatBoost stopped at {cat_model.get_best_iteration()} iterations "
      f"(ceiling 2000, patience 100, monitoring validation AUC).")

cat_val_pred  = cat_model.predict(X_val).astype(int)
cat_val_proba = cat_model.predict_proba(X_val)[:, 1]
validation_results.append(evaluate_model("CatBoost", y_val, cat_val_pred, cat_val_proba))
print("\nValidation performance:")
print(pd.Series(validation_results[-1]))

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.pipeline import Pipeline
from catboost import CatBoostClassifier # Import CatBoostClassifier

# --- Stacking meta-ensemble: 5 heterogeneous base learners, logistic meta-learner ---
# Base learners are re-instantiated fresh (not the fitted models above) because
# StackingClassifier must refit each of them on internal out-of-fold partitions of the
# TRAINING set; passing already-fitted models would leak their full-train fit.
stack_base_learners = [
    ("logreg", Pipeline([                      # LR needs scaling; trees do not
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=2000,
                                   random_state=RANDOM_STATE)),
    ])),
    ("rf",   RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                    class_weight="balanced",
                                    random_state=RANDOM_STATE, n_jobs=-1)),
    ("xgb",  xgb.XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8,
                               scale_pos_weight=scale_pos_weight,
                               eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)),
    ("lgbm", lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                                scale_pos_weight=scale_pos_weight,
                                random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)),
    ("cat",  CatBoostClassifier(iterations=400, learning_rate=0.05, depth=6,
                                scale_pos_weight=scale_pos_weight,
                                random_seed=RANDOM_STATE, verbose=0)),
]

stacking_model = StackingClassifier(
    estimators=stack_base_learners,
    final_estimator=LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    stack_method="predict_proba",   # meta-learner sees calibrated probabilities
    n_jobs=1,
)
print("Fitting stacking ensemble (5 base learners x 5 out-of-fold partitions + final refits)...")
stacking_model.fit(X_train, y_train)

stack_val_pred  = stacking_model.predict(X_val)
stack_val_proba = stacking_model.predict_proba(X_val)[:, 1]
validation_results.append(evaluate_model("Stacking Ensemble (5 learners)",
                                         y_val, stack_val_pred, stack_val_proba))
print("\nValidation performance:")
print(pd.Series(validation_results[-1]))

# Meta-learner weights: how the stack allocates trust across its base learners
meta_w = pd.Series(stacking_model.final_estimator_.coef_[0],
                   index=[name for name, _ in stack_base_learners])
print("\nMeta-learner (logistic) coefficients on base-learner probabilities:")
print(meta_w.round(3).sort_values(ascending=False))

In [ ]:
# --- Soft-voting blend: unweighted average of the three boosting models' probabilities ---
# No refitting is required; this is a pure probability-space combination and serves as a
# minimal-complexity benchmark against which the stacking meta-learner must justify itself.
blend_val_proba = (xgb_val_proba + lgbm_val_proba + cat_val_proba) / 3.0
blend_val_pred  = (blend_val_proba >= 0.5).astype(int)
validation_results.append(evaluate_model("Soft-Vote Blend (XGB+LGBM+Cat)",
                                         y_val, blend_val_pred, blend_val_proba))
print("Validation performance:")
print(pd.Series(validation_results[-1]))

In [ ]:
val_results_all = pd.DataFrame(validation_results).drop_duplicates(subset=["Model"], keep="last") \
                    .set_index("Model") \
                    .sort_values("ROC-AUC", ascending=False)
print("=== VALIDATION LEADERBOARD (§2.4 + §2.5 + §2.7 models) — "
      "10-Metric Suite) ===")
val_results_all.style.format("{:.4f}").background_gradient(cmap="Greens", axis=0)

### 2.7.5 Interpretation of the Validation Leaderboard

**The advanced models take the lead, narrowly:** The scores for the advanced models are very close but CatBoost has the highest validation ROC-AUC among all models in the notebook (0.7803), followed by soft-vote blend (0.7795), Random Forest (0.7779), stacking ensemble (0.7771) and LightGBM (0.7757). This compression of the top of the table is noteworthy in itself: The predictive signal from this dataset is basically in a few behavioural features (§2.2.4, §2.6) and once any competent learner has picked up the `PAY_0`/`MONTHS_DELINQUENT` structure, little headroom remains.

**Two boosting philosophies, two stopping points:** CatBoost took 266 iterations to reach its stopping point (when the validation AUC stopped improving), while LightGBM reached a stopping point after only 20 iterations. This is not a bug; it's just a result of the design of LightGBM: LightGBM's leaf-wise growth extracts the dominant signal very quickly and then begins to overfit the validation metric, so it stops early. CatBoost's depth-limited symmetric trees learn more slowly, and its ordered boosting resists overfitting the validation metric, so it trains far longer.

The real-life downside of the short fit of LightGBM is not the loss of discrimination, its AUC and KS are not that bad, but its recall drops to 0.295 at the default 0.5 cutoff. The Youden diagnostic above shows that at the discrimination-optimal threshold of 0.346, recall recovers to 0.625 at 0.789 specificity, in line with the class-weighted models.

This is the same issue Cao et al. (in the review) address with Youden's index, and it anticipates the calibration discussion in §2.9, which is an operating choice, not related to model quality.


**What the meta-learner reveals:** The weights of the stacking meta-coefficients are not equal and are telling: Random Forest (2.09), CatBoost (1.86), logistic base learner (0.62) and XGBoost (0.004). The near zero value of XGBoost weight doesn't imply that XGBoost is not informative at all but it means that OOF probabilities of XGBoost are close to a linear combination of LightGBM and CatBoost probabilities, thus bringing little new to the table that the meta-learner can't already achieve. The weight instead goes to the learner that makes the most errors that are "decorrelated" from the boosting family: the bagging-based Random Forest. This is a real demonstration of ensemble-diversity which Ayari et al. state to be the mechanism behind the superiority of hybrid model: Heterogeneity of error not the number of base learners is what an ensemble monetises.

**A note on the free lunch:** This soft-vote combination, none of which needed extra training, is equivalent to the 25-fits stacking ensemble, on the third decimal place of the validation AUC. The stack thus needs to be justified on the held-out test set and the calibration set.

On parsimony grounds the blend is favoured; §2.9 tests whether the held-out data agrees.


---
## 2.8 Model Explainability: SHAP Analysis of the Gradient-Boosting Model

`TreeExplainer` computes exact SHAP values for tree ensembles in polynomial time, so the
analysis below runs on the fitted LightGBM model over a 2,500-row stratified sample of the
validation partition. Two views are produced: the **beeswarm plot** (each dot is one
borrower-feature pair, coloured by feature value, positioned by SHAP contribution) reveals
the direction and shape of each effect; the **mean |SHAP| bar chart** gives the global
importance ranking on the log-odds scale.

In [ ]:
import warnings
warnings.filterwarnings("ignore")   # silence SHAP/tqdm Future/UserWarnings in submission output
import shap

# Stratified 2,500-row explanation sample from the validation partition
X_val_df = pd.DataFrame(X_val, columns=FEATURE_COLS) if not isinstance(X_val, pd.DataFrame) else X_val
shap_sample = X_val_df.groupby(y_val.values, group_keys=False) \
                      .apply(lambda g: g.sample(frac=2500 / len(X_val_df),
                                                random_state=RANDOM_STATE))
print(f"SHAP explanation sample: {len(shap_sample):,} validation rows "
      f"({y_val.loc[shap_sample.index].mean()*100:.1f}% default, stratified)")

explainer = shap.TreeExplainer(lgbm_model)
shap_vals = explainer.shap_values(shap_sample)
if isinstance(shap_vals, list):            # older SHAP returns [class0, class1]
    shap_vals = shap_vals[1]

# Global importance table (mean absolute SHAP value, log-odds scale)
shap_importance = pd.Series(np.abs(shap_vals).mean(axis=0), index=shap_sample.columns) \
                    .sort_values(ascending=False)
print("\nTop 10 features by mean |SHAP| (log-odds contribution):")
print(shap_importance.head(10).round(4))

fig = plt.figure(figsize=(9, 7))
shap.summary_plot(shap_vals, shap_sample, max_display=15, show=False)
plt.title("Figure 2.11: SHAP Beeswarm: LightGBM Default-Risk Attributions",
          fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("fig2_11_shap_beeswarm.png", bbox_inches="tight")
plt.show()

fig = plt.figure(figsize=(8, 6))
shap.summary_plot(shap_vals, shap_sample, plot_type="bar", max_display=15, show=False)
plt.title("Figure 2.12: SHAP Global Importance (mean |SHAP|)",
          fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("fig2_12_shap_bar.png", bbox_inches="tight")
plt.show()

### 2.8.1 Interpretation of the SHAP Attributions

**Three attribution methods, one verdict.** The SHAP global ranking is led by `MONTHS_DELINQUENT` (mean |SHAP| 0.183) and `AVG_PAY_DELAY` (0.120), followed by `PAY_0` (0.064), `UTILISATION_RATIO` (0.056) and `LIMIT_BAL` (0.051). All three methodologically independent importance measures used in this notebook (impurity-based rankings from the Random Forest and XGBoost in §2.4.7, maximum-likelihood z-statistics in §2.6, and Shapley attributions here) agree that recent repayment behaviour is the dominant driver of default risk.

The most solid evidence of robustness an explainability analysis can provide is triangulation across attribution families, as each family of methods has distinct known biases which do not overlap between families.


**Direction, not just magnitude.** The beeswarm plot also validates the fact that all the main effects act in the economic sense, high (red) delinquency values drive a prediction towards the default, high credit limits and large recent payment amounts (`PAY_AMT1` and `PAY_AMT2`) drive a prediction away from the default, and higher utilisation increases risk. These signs are similar to the econometric coefficient signs in §2.6.2, meaning that the black-box model has learned the same economics as the transparent one, but with some non linear refinements.

**Feature engineering vindicated:** Two of these four features were two of the most influential features: `MONTHS_DELINQUENT` and `AVG_PAY_DELAY` were not present in the raw schema, but were created in §2.2.6. This SHAP ranking is direct evidence that the feature-engineering effort, not model sophistication alone, accounts for a significant share of the models' discrimination.


**The fair-lending reading.** The last place in the roster of the demographic features (SEX, EDUCATION, MARRIAGE, AGE) reflects their small (but significant) econometric effects. The model decisions made by deployed model are conduct-based, the desired approach for a credit model that is subject to GDPR Article 22 rights pertaining to automated decision making and is classified as a high risk use case in Annex III of the EU AI Act [8]. SHAP values are computed per applicant, so the same machinery that produced the global plots above also provides the individual adverse-action explanation to which a declined applicant is entitled.


**One caution.** SHAP explains the model, not the world: the attributions quantify the use of each feature by the fitted LightGBM, and causal claims about the borrowers remain in the econometric analysis (of §2.6) and in the theory (of §2.10).

---
## 2.9 Final Held-Out Test Benchmark, Bootstrap Confidence Intervals, and Model Selection

We now test both advanced models on the same identical 20% test partition which we used in the benchmark testing of the §2.4 and §2.5 models, and come up with a single, consolidated benchmark of the nine models on the full 10-metric suite. However, a model selection decision cannot be made based on point estimates only: two AUCs, just a few thousandths apart, may not be statistically distinguishable, with a test set of 6,000 rows. Following standard scorecard-validation practice, after scoring the test set a stratified bootstrap (1,000 resampled test sets) is used to generate a 95% confidence interval around the ROC-AUC of each model, allowing the final ranking to not only indicate which model is in the lead, but also to determine whether that lead is significant at this sample size.

In [ ]:
# --- Score the advanced models on the held-out test partition ---
lgbm_test_proba  = lgbm_model.predict_proba(X_test)[:, 1]
lgbm_test_pred   = lgbm_model.predict(X_test)
cat_test_proba   = cat_model.predict_proba(X_test)[:, 1]
cat_test_pred    = cat_model.predict(X_test).astype(int)
stack_test_proba = stacking_model.predict_proba(X_test)[:, 1]
stack_test_pred  = stacking_model.predict(X_test)
blend_test_proba = (xgb_test_proba + lgbm_test_proba + cat_test_proba) / 3.0
blend_test_pred  = (blend_test_proba >= 0.5).astype(int)

advanced_test_results = [
    evaluate_model("LightGBM",                       y_test, lgbm_test_pred,  lgbm_test_proba),
    evaluate_model("CatBoost",                       y_test, cat_test_pred,   cat_test_proba),
    evaluate_model("Stacking Ensemble (5 learners)", y_test, stack_test_pred, stack_test_proba),
    evaluate_model("Soft-Vote Blend (XGB+LGBM+Cat)", y_test, blend_test_pred, blend_test_proba),
]

grand_test_df = pd.concat([test_results_df,
                           pd.DataFrame(advanced_test_results).set_index("Model")]) \
                  .sort_values("ROC-AUC", ascending=False)

print("=== GRAND FINAL: 9-MODEL HELD-OUT TEST BENCHMARK — "
      "10-Metric Suite ===")
grand_test_df.style.format("{:.4f}").background_gradient(cmap="Greens", axis=0)

In [ ]:
# --- ROC curves: full nine-model field on the held-out test set ---
all_test_probas = {
    "Logistic Regression":            lr_test_proba,
    "Random Forest":                  rf_test_proba,
    "XGBoost (Gradient Boosting)":    xgb_test_proba,
    "Support Vector Machine (RBF)":   svm_test_proba,
    "Deep Neural Network":            dnn_test_proba,
    "LightGBM":                       lgbm_test_proba,
    "CatBoost":                       cat_test_proba,
    "Stacking Ensemble (5 learners)": stack_test_proba,
    "Soft-Vote Blend (XGB+LGBM+Cat)": blend_test_proba,
}

fig, ax = plt.subplots(figsize=(9.5, 8))
for name, proba in sorted(all_test_probas.items(),
                          key=lambda kv: -roc_auc_score(y_test, kv[1])):
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, linewidth=1.8,
            label=f"{name} (AUC={roc_auc_score(y_test, proba):.4f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random (AUC=0.5000)")
ax.set_title("Figure 2.13: ROC Curves, All Nine Models (Held-Out Test Set)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right", fontsize=8.5)
plt.tight_layout()
plt.savefig("fig2_13_roc_all_models.png", bbox_inches="tight")
plt.show()

In [ ]:
# --- Stratified bootstrap: 95% confidence intervals on test ROC-AUC ---
N_BOOT = 1000
rng = np.random.default_rng(RANDOM_STATE)
y_test_arr = np.asarray(y_test)
pos_idx = np.flatnonzero(y_test_arr == 1)
neg_idx = np.flatnonzero(y_test_arr == 0)

boot_rows = []
for name, proba in all_test_probas.items():
    proba = np.asarray(proba)
    aucs = np.empty(N_BOOT)
    for b in range(N_BOOT):
        # Stratified resample: preserve the 22.1% class ratio in every replicate
        idx = np.concatenate([rng.choice(pos_idx, size=len(pos_idx), replace=True),
                              rng.choice(neg_idx, size=len(neg_idx), replace=True)])
        aucs[b] = roc_auc_score(y_test_arr[idx], proba[idx])
    lo, hi = np.percentile(aucs, [2.5, 97.5])
    boot_rows.append({"Model": name,
                      "Test ROC-AUC": roc_auc_score(y_test_arr, proba),
                      "Bootstrap Mean": aucs.mean(),
                      "95% CI Lower": lo, "95% CI Upper": hi,
                      "CI Width": hi - lo})

boot_df = pd.DataFrame(boot_rows).set_index("Model") \
            .sort_values("Test ROC-AUC", ascending=False)
print(f"=== BOOTSTRAP ROC-AUC CONFIDENCE INTERVALS "
      f"({N_BOOT} stratified resamples of the test set) ===")
display(boot_df.round(4))

fig, ax = plt.subplots(figsize=(9, 5.5))
order = boot_df.index[::-1]
ax.errorbar(boot_df.loc[order, "Test ROC-AUC"], range(len(order)),
            xerr=[boot_df.loc[order, "Test ROC-AUC"] - boot_df.loc[order, "95% CI Lower"],
                  boot_df.loc[order, "95% CI Upper"] - boot_df.loc[order, "Test ROC-AUC"]],
            fmt="o", color="#1a6b3c", ecolor="#7fbf9b", elinewidth=2.5,
            capsize=4, markersize=6)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=9)
ax.set_xlabel("ROC-AUC (point estimate with bootstrap 95% CI)")
ax.set_title("Figure 2.14: Test ROC-AUC with Bootstrap 95% Confidence Intervals",
             fontsize=12, fontweight="bold")
ax.grid(axis="x", alpha=0.4)
plt.tight_layout()
plt.savefig("fig2_14_bootstrap_auc_ci.png", bbox_inches="tight")
plt.show()

### 2.9.4 Interpretation, Model Selection, and Regulatory Synthesis

**The literature's prediction holds on the held-out test set.** The top three ensembles are the soft-vote ensemble (ROC-AUC 0.7930), the CatBoost (0.7921) and the stacking ensemble (0.7916) all of which are based on §2.7, ahead of the best §2.4 model (XGBoost, 0.7891), the Deep Neural Network (0.7882) and Random Forest (0.7875). This replicates, on a pristine sample, the main empirical finding of Ayari et al. (2026): ensemble/hybrid classifiers are always better than single classifiers for tabular credit data. The margin is not dramatic (+0.4 to +0.9 AUC points over the incumbent champion), but rather honest – which is the improvement scale their review reports for mature benchmark datasets.

**Different metrics crown different models, exactly as §2.4.1 warned.** CatBoost is the number one in the KS statistic, the industry's separation metric. The G-Mean (0.7202) and recall (0.6451) of XGBoost remain the best and the second best, respectively. The stacking ensemble is the notebook's best Brier score (0.1321) because of the built-in calibration effect of the logistic meta-learner over the base models' probabilities. LightGBM is the lowest Type I error (0.0372) simply because the compressed probabilities hardly ever go over the
The test-set form of the §2.7.5 threshold artifact, which is 0.5 cutoff. None of the models are the clear leader in all 10 of the metrics, justifying the multi-metric evaluation design.


**What the bootstrap adds: statistical honesty.** The 95% confidence intervals of the top 7 models are quite broad, with each being approximately ±0.014 wide, thus the advantage of the blend over CatBoost, the stack or even XGBoost is not statistically significant on a 6,000 row test set. Two conclusions, however, can be determined, and are not just numerical: Logistic Regression (upper bound 0.7782) is at or below the lower bound of the leading ensembles and the SVM (upper bound 0.7698) is also at or below the lower bound of the leading ensembles; thus, the gradient-boosting family's advantage over the linear and kernel baselines is statistically, rather than only numerically supported. It is the norm that a model validation committee would set that the ranking with its uncertainty is reported, instead of a winner being declared on the third decimal.

**Model selection recommendation.** All evidence considered, I recommend using CatBoost as the production candidate – it was just as discriminatory as the blend within noise, it has the best KS of all the models, it is a single model not three (or eight) to govern, monitor and serve, it is fully compatible with the per-applicant SHAP explanation machinery of §2.8. The “soft-vote” blend is the natural low cost challenger model. The advantage of the stacking ensemble suggests that the deployment rule is rather than which model to use: any model used for IFRS 9 provisioning should be post-hoc calibrated (Platt scaling or isotonic regression, fitted on validation partition) since the provisioning consumes the probability, rather than the rank.

**The financial materiality of the gain.** IFRS 9's ECL is PD x LGD x EAD, which means that the quality of the PD model propagates linearly into provisions, which is a critical factor. These ensemble gains of roughly +0.03 KS and up to +0.9 AUC points over the traditional champion were achieved with a consistent methodology and no additional data.
The closer the gap between good and bad cards at each cutoff, the more precisely an issuer can set its cutoff to reduce Type II errors
(rig the system, defaulters approved) while increasing the number of cards they approve at a fixed risk appetite (fewer Type I errors: good customers rejected). The trade-off of CatBoost vs. LightGBM is presented in the Type I/Type II columns of the grand table: At the default cutoff, CatBoost makes 35.6% of mistakes, LightGBM makes 67.3% of mistakes, but they discriminate almost similarly (by AUC). Once the model is selected based on discrimination and calibration criteria, the cut-off (that is, pricing and risk appetite decision) will be set by a tool such as the Youden analysis of §2.7, or by a cost-weighted threshold (§2.10.4). That selection process, with its three-way split, single pass at the test set, and bootstrap uncertainty on the final ranking, is defensible to a regulator, an auditor, and an internal model-risk function.



---
## 2.10 Theory of Credit Card Profitability and Loan Provisioning

### 2.10.1 The Economic Architecture of Credit Card Profitability

A credit card issuer derives revenue from two very different economic activities, and the first step towards relating the models in this notebook to actual banking is to differentiate these two activities:

**(i) The transaction function** is the first of the two revenue streams, and that is through the interchange fee or about 2.0%–2.25% of the value of the transaction in the United States (which depends on the type and details of the card, merchant category, and whether the card is present or not) paid by the merchant's acquiring bank to the merchant's issuing bank on every purchase. The visible costs for the operation of a card programme, such as the cost of cardholder rewards, fraud prevention and network/processing infrastructure are supported by interchange revenue.

**(ii) The credit function** is where revenue is generated by means of net interest margin (NIM) on revolving balances (those amounts that a cardholder does not pay back in full each billing cycle), which the credit card issuer charges interest, often at a high rate of 18%–29% annual percentage rate (APR), but borrows at a much lower cost of funds, with the spread going to the credit card issuer as long as the cardholder continues to service the balance and does not default.

**A critical empirical fact that should reshape how a student approaches this section's analysis:**
the Federal Reserve itself estimates the credit component of the overall credit card profitability to be about 80% [9] while the transaction component is slightly negative on average; that is, the interchange revenues generated by issuers, on average, are less than rewards expenses and other transactional costs borne by issuers. A further ~15% of profitability is due to late fees. The clear implication: Credit risk management, the ability to distinguish those revolving borrowers who will pay back from those who will default isn't an ancillary risk-control discipline that can be cramped into a transaction-fee business, but is the primary determinant of whether a credit card portfolio will be profitable at all. The direct applicability of these machine learning models, developed in §2.3–§2.5, to the theory of credit card profitability is an idea which will be developed throughout the rest of this section, and which turns on this single fact.


### 2.10.2 Revolving Credit Dynamics and the "Revolver vs. Transactor" Distinction

The above profitability architecture suggests a segmentation of behaviours that the features and characteristics of this dataset, as themselves, map onto:

- **"Transactors"** are those that pay their statement balance in full every month, thus not earning any interest income for the issuer and little or negative net profitability per account (per §2.10.1) even though they are "safe".

- **"Revolvers"** carry a balance from month to month, generating net interest margin for the issuer for as long as they keep servicing the balance. This is the source of most issuer profits. A revolver who stops paying is no longer a profitable revolver but a defaulting credit exposure.

In economic substance, these features (MONTHS_DELINQUENT, AVG_PAY_DELAY, UTILISATION_RATIO and PAYMENT_TO_BILL_RATIO) created by the engineering (§2.2.6) are actually features of the population the bank's profitability hinges most heavily on, which is why the feature-importance (§2.4.7) result showed them dominating both ensemble models' decisions. The results in §2.2.4 (default risk rising sharply with recent delinquency) are the credit-risk mirror image of the results in §2.10.1 (profitability concentrated in revolving balances); the same revolving-balance population that provides ~80% of the profit when performing, is the population whose behaviour the models in this notebook are designed to monitor for early signs of non-performance.


### 2.10.3 Connecting the Models to IFRS 9 Expected Credit Loss and Loan Provisioning

International Financial Reporting Standard 9 (IFRS 9) prescribes that the banks use the forward-looking provision, called Expected Credit Loss (ECL) instead of the more traditional approach of 'incurred loss', and consist of three parts that closely match the three parts of this notebook's modelling pipeline:

| IFRS 9 Component | Definition | This Notebook's Connection |
|---|---|---|
| **Probability of Default (PD)** | The likelihood a borrower defaults within a defined horizon (here, "next month") | **Exactly what every model in §2.4–§2.5 outputs.** `predict_proba()`'s second column for every traditional ML model, and the DNN's sigmoid output, *are* PD estimates at the individual-account level. |
| **Exposure at Default (EAD)** | The expected credit exposure at the point of default | Directly observable from this dataset's `LIMIT_BAL` and current `BILL_AMTx` fields — the EDA in §2.2.5 specifically distinguished EAD-type exposure-size features (weakly correlated with default) from PD-type behavioural features (strongly correlated with default), which is itself the practical reason these two components are estimated separately in real provisioning models rather than conflated into one figure. |
| **Loss Given Default (LGD)** | The proportion of exposure not recovered after default (1 − recovery rate) | Not directly estimable from this dataset (which has no post-default recovery/collections data), but conceptually the third multiplier: `ECL ≈ PD × EAD × LGD`, aggregated across the portfolio. |

The practical loan-provisioning workflow this notebook's models would feed into a bank's finance
function is therefore: score every active account's PD using the recommended production candidate from §2.9.4 (CatBoost, post-hoc calibrated as discussed there); (2) combine each account's PD with its
EAD (current exposure) and a portfolio- or segment-level LGD assumption; (3) sum across the
portfolio to set the **provision**: capital the bank must hold against expected, not yet
realised, losses. A model such as this notebook's Random Forest, with a test-set ROC-AUC of
~0.79 (§2.5.4), allows the bank to rank-order its entire portfolio by forward-looking default
risk and **provision more precisely** than a model relying on demographic information alone
(§2.2.2's finding that demographic effects are individually weak) or on raw exposure size alone
(§2.2.5's finding that `BILL_AMT` is a poor standalone predictor), both of which were
historically more available to lenders than today's full repayment-history behavioural data, and
both of which this notebook's EDA and modelling results show to be inferior signals.

### 2.10.4 Automated Credit Scoring and Origination

As mentioned in §2.2.3, LIMIT_BAL is already the result of an underwriting decision. In production, a bank would use a PD model very similar to those constructed in §2.4 and §2.5, but trained on a slightly different feature set lacking much of the post-origination behavioural history that this notebook makes full use of. Such a model would be used to:

- Determine the starting credit limit when the account is opened, consider risk (higher = higher EAD) vs interchange and interest income (higher = greater growth).
- Continuous, ongoing decisions to increase or decrease a user's limit (increases to revolvers that do not present a high risk to the system, or perform well; decreases or freezes on revolvers whose `PAY_x` trajectory changes over time). This is the same rank ordering signal that this notebook's models output, but occurs continuously over time rather than at origination.
- Conduct a model-governance/fair-lending review, which, as noted in the finding in this notebook (§2.2.2, §2.4.7), explicitly considers the importance (or lack of importance) of variables that are adjacent to protected characteristics, such as demographic variables like `SEX`, `EDUCATION`, `MARRIAGE`, or `AGE` (directly relevant to the finding in this notebook).



### 2.10.5 Safeguarding Portfolio Profitability: Tying the Threads Together

To recap the credit function of the credit card (which generates most of its profitability, not the transaction function) and the fact that profitability all depends on the continued performance of the revolving borrowers, the most important investment a card issuer could make to protect profitability in its portfolio is actually the type of behavioural default-risk model developed in this notebook, not, as one might think from focusing on transaction volume, growth in spend volume. In §2.5.4 and §2.9 we saw a range of roughly 0.01 to 0.02 in test-set ROC-AUC between the best and worst models. Applied to a portfolio of hundreds of thousands or millions of revolving accounts, that difference translates into a materially different set of accounts targeted for proactive intervention (a credit-limit freeze, a hardship-programme offer, an early collections contact) before default rather than after. That, in essence, is the economic rationale for the switch from incurred-loss to expected-loss (IFRS 9) provisioning in the first place.

It is this direct, substantive connection between the empirical machine-learning findings and the economic theory of how a credit card portfolio makes or loses money that this section was asked to draw.

---
# References

[1] U.S. Congress, "S.1582, GENIUS Act of 2025, 119th Congress," Congress.gov, 2025. [Online]. Available: https://www.congress.gov/bill/119th-congress/senate-bill/1582/text

[2] Harvard Law School Forum on Corporate Governance, "SEC enforcement: 2025 year in review," Jan. 2026. [Online]. Available: https://corpgov.law.harvard.edu/2026/01/21/sec-enforcement-2025-year-in-review/

[4] I.-C. Yeh and C.-H. Lien, "The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients," *Expert Systems with Applications*, vol. 36, no. 2, pp. 2473-2479, 2009.

[5] H. Ayari, R. Guetari, and N. Kraiem, "Machine learning powered financial credit scoring: a systematic literature review," *Artificial Intelligence Review*, vol. 59, no. 1, art. 13, 2026, doi: 10.1007/s10462-025-11416-2.

[6] S. M. Lundberg and S.-I. Lee, "A unified approach to interpreting model predictions," in *Advances in Neural Information Processing Systems 30*, 2017, pp. 4765-4774.

[7] European Union, "Regulation (EU) 2016/679 (General Data Protection Regulation), Article 22," *Official Journal of the European Union*, 2016.

[8] European Union, "Regulation (EU) 2024/1689 (Artificial Intelligence Act), Annex III," *Official Journal of the European Union*, 2024.

[9] R. M. Adams, V. M. Bord, and B. Katcher, "Credit card profitability," *FEDS Notes*, Board of Governors of the Federal Reserve System, Washington, DC, Sep. 2022.

[10] International Accounting Standards Board, *IFRS 9 Financial Instruments*, London, U.K.: IFRS Foundation, 2014.